# Implementing Transformer Architecture: A Step-by-Step Guide


- Key sections:
  - 3.1: Encoder and Decoder Stacks
  - 3.2: Attention Mechanism
  - 3.3: Position-wise Feed-Forward Networks
  - 3.4: Embeddings and Softmax
  - 3.5: Positional Encoding
  - 5.4: Regularization (dropout strategy)

## Implementation Strategy
Breaking down the architecture into manageable pieces and gradually adding complexity:

1. Start with foundational components:
   - Embedding + Positional Encoding
   - Single-head self-attention

2. Build up attention mechanism:
   - Extend to multi-head attention
   - Add cross-attention capability
   - Implement attention masking

3. Construct larger components:
   - Encoder (self-attention + FFN)
   - Decoder (masked self-attention + cross-attention + FFN)

4. Combine into final architecture:
   - Encoder-Decoder stack
   - Full Transformer with input/output layers

## Development Tips
1. Visualization and Planning:
   - Draw out tensor dimensions on paper
   - Sketch attention patterns and masks
   - Map each component back to paper equations
   - This helps catch dimension mismatches early!

2. Dimension Cheat Sheet:
   - Input tokens: [batch_size, seq_len]
   - Embeddings: [batch_size, seq_len, d_model]
   - Attention matrices: [batch_size, num_heads, seq_len, seq_len]
   - FFN hidden layer: [batch_size, seq_len, d_ff]
   - Output logits: [batch_size, seq_len, vocab_size]

3. Common Pitfalls:
   - Forgetting to scale dot products by √d_k
   - Applying mask too early or too late
   - Incorrect mask dimensions or application
   - Missing residual connections
   - Wrong order of layer norm and dropout
   - Tensor dimension mismatches in attention
   - Not handling padding properly

4. Performance Considerations:
   - Memory usage scales with sequence length squared
   - Attention computation is O(n²) with sequence length
   - Balance between d_model and num_heads
   - Trade-off between model size and batch size

## Testing Strategy
- Test each component independently
- Verify shape preservation
- Check attention patterns
- Confirm mask effectiveness
- Validate gradient flow
- Monitor numerical stability

Remember: The key to successfully implementing the Transformer is understanding how each piece fits together and maintaining clear dimension tracking throughout the implementation.

In [ ]:
import torch
from torch import nn

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

In [ ]:
# device = ["cuda","mps","cpu"]

In [ ]:
# a utility for calculating running average
class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


def averager(func):
    num = 0
    total = 0

    # val = 0.0

    @wraps(func)
    def inner(val, sz):
        nonlocal num
        nonlocal total
        num += val * sz
        total += sz
        print(num, total, num / total)
        return num, total, num / total

    return inner

In [ ]:
@averager
def calc_product(n, a):
    return n, a


print(averager(calc_product(2, 2)))

print(calc_product(2, 3))

# Problem 2: Implement a Transformer

## Part 2.A

## Attention Head

In [ ]:
import copy
import math

torch.manual_seed(42)


class AttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int):
        # dim: the dimension of the input
        # n_hidden: the dimension of the keys, queries, and values

        super().__init__()

        self.W_K = nn.Linear(dim, n_hidden)  # W_K weight matrix
        self.W_Q = nn.Linear(dim, n_hidden)  # W_Q weight matrix
        self.W_V = nn.Linear(dim, n_hidden)  # W_V weight matrix
        self.n_hidden = n_hidden
        self.dim = dim
        # self.temp = torch.tensor([])
        # self.temp2 = torch.tensor([])
        # self.temp3 = torch.tensor([])

        print(self.count_params())
        # self.calculate_shapes()

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

    # def calculate_shapes(self):
    #     print("Attention Matrix shape",self.temp.shape)
    #     print(f"Before Normalizing Attention Head: {self.temp2.shape}")
    #     print(f"Attention Head final shape: {self.temp3.shape}")

    def calc_attention_mask(
        self, inp: torch.Tensor, mask: Optional[torch.Tensor] = None
    ):
        """
        returns fills inp with -infinity where attention mask is 0..
        so that after softmax output is zero
        """
        if not mask:
            return inp

        return inp.masked_fill(mask == 0, -float("inf"))

    # def forward(
    #     self, x: torch.Tensor, attn_mask: Optional[torch.Tensor]=None
    # ) -> Tuple[torch.Tensor, torch.Tensor]:
    #     """
    #     # x                the inputs. shape: (B x T x dim)
    #     # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
    #     #                  contains 1 if (in batch b) token i should attend on token j and 0
    #     #                  otherwise. shape: (B x T x T)
    #     #
    #     # Outputs:
    #     # attn_output      the output of performing self-attention on x. shape: (Batch x Num_tokens x n_hidden)
    #     # alpha            the attention weights (after softmax). shape: (B x T x T)
    #     #
    #     """
    #     print(f"Attention Mask:{attn_mask}")
    #
    #     A, Z = None, None
    #     """
    #     #todo : Compute self attention on x.
    #     #       (1) First project x to the query Q, key K, value V.
    #     #       (2) Then compute the attention weights alpha as:
    #     #                  alpha = softmax(QK^T/sqrt(n_hidden))
    #     #           Make sure to take into account attn_mask such that token i does not attend on token
    #     #           j if attn_mask[b, i, j] == 0. (Hint, in such a case, what value should you set the weight
    #     #           to before the softmax so that after the softmax the value is 0?)
    #     #       (3) The output is a linear combination of the values (weighted by the alphas):
    #     #                  out = alpha V
    #     #       (4) return the output and the alpha after the softmax
    #
    #     # ======= Answer START ========
    #     """
    #     print(f"x={type(x)},{x.shape if type(x) is torch.tensor else len(x)} in attention head")
    #     Q = self.W_Q(x).T
    #     K = self.W_K(x).T
    #     V = self.W_V(x).T
    #     print(f"Without Transpose:{self.W_Q(x).shape}")
    #     print(f"{Q.shape=},{K.shape=},{V.shape=}")
    #     #print(f"{(Q@K.T).shape=}\n\n\n-------------")
    #     #print(f"{torch.matmul(Q,K.T).shape}")
    #     A = torch.matmul(Q,K.transpose(dim0=1,dim1=2))
    #     print(f"{A.shape=}")
    #     self.temp = A
    #     #print(A.shape) # dim*dim
    #     #todo -> causal masking auto regressive.. for time based
    #     # if not attn_mask:
    #     #     out = alpha
    #     # else:
    #     #     first = attn_mask[0,:,1]
    #     #     second = attn_mask[0,:,2]
    #     #     for i in first:
    #     #         for j in second:
    #     #             if j==0:
    #     #                 out=-alpha
    #
    #     #mask = self.calc_attention_mask(A,attn_mask)
    #     #print(self.calculate_shapes())
    #     if attn_mask is not None:
    #         Z=A+attn_mask
    #     else:
    #         Z=A
    #     self.temp2 = Z
    #     #print(f"Masking score: {inp.shape}")
    #
    #     Z = torch.softmax(Z/n_hidden**.5,dim=-1)
    #     print(f"before multiplication{Z.shape=}")
    #     Z = torch.matmul(Z,V)
    #     #print(f"Checking shape along dim 1: {torch.softmax(Z,dim=1).shape}")
    #     print(f"{Z.shape=}")
    #
    #     self.temp3 = Z
    #     #self.calculate_shapes()
    #
    #
    #
    #
    #
    #
    #     #attn_output = torch.softmax(attn_scores,dim=-1)
    #
    #     # ======= Answer  END ========
    #     print("Attention Head Successful!")
    #     print(f"Attention output shape={A.shape}, Z={Z.shape} calculated for attention head\n\n------------")
    #
    #     return A,Z

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # x shape: (B, T, dim)
        print("\n\n-----------------------\nIn Attention Head:\n\n=")
        print(f"{x.shape=}")
        # 1. Project inputs to Q, K, V (Keep 3D structure intact)
        Q = self.W_Q(x)  # (B, T, n_hidden)
        K = self.W_K(x)  # (B, T, n_hidden)
        V = self.W_V(x)  # (B, T, n_hidden)
        print(f"{Q.shape=},{K.shape=},{V.shape=}")
        # 2. Compute attention scores matrix
        # Transpose the last two dimensions of K to match (B, n_hidden, T)
        # A = torch.matmul(Q, K.transpose(-2, -1))  # Shape: (B, T, T)
        # print(f"{A.shape=}")
        # scores = A
        A = Q @ K.transpose(-2, -1)
        # print(f"Second way {A.shape=}")
        # 3. Apply the Attention Mask safely before Softmax

        if attn_mask is not None:
            # Everywhere mask is 0, fill scores with a massive negative number
            A = A.masked_fill(attn_mask == 0, float("-inf"))

        # 4. Normalize and apply Softmax to get Alpha weights
        # doing layer by layer so in each layer n_hidden dimensions
        A = torch.softmax(A / math.sqrt(self.n_hidden), dim=-1)  # Shape: (B, T, T)

        # 5. Compute the final context vector
        # (B, T, T) x (B, T, n_hidden) -> (B, T, n_hidden)
        Z = A @ V

        # 6. Return strictly in the order requested by your docstring/parent class
        return Z, A

    # def backward(self):

### Debug

In [ ]:
import torch
import torch.nn as nn

# 1. Define the dimensions
BATCH_SIZE = 10
SEQ_LEN = 100
INPUT_DIM = 64
OUTPUT_DIM = 32

# 2. Instantiate the linear layer
# It expects the incoming features to be 64, and transforms them into 32
linear_layer = nn.Sequential(
    nn.Linear(in_features=INPUT_DIM, out_features=1),
    nn.Linear(in_features=1, out_features=OUTPUT_DIM),
)

# 3. Create a 3D input tensor matching your Transformer data layout
# Shape: (Batch Size, Sequence Length, Features)
x = torch.randn(BATCH_SIZE, SEQ_LEN, INPUT_DIM)
print(f"Input shape:  {x.shape}")  # Output: torch.Size([10, 100, 64])

# 4. Pass the tensor through the layer
output = linear_layer(x)
print(f"Output shape: {output.shape}")  # Output: torch.Size([10, 100, 32])

In [ ]:
x.transpose(-2, -1).shape

In [ ]:
attention = AttentionHead(INPUT_DIM, OUTPUT_DIM)
attention.count_params()
Z, A = attention(x)

In [ ]:
Z.shape

In [ ]:
A.shape

In [ ]:
x = torch.tensor([[2.0, 3]] * 3)
x

In [ ]:
inp = torch.randn(10, 100, 64)

In [ ]:
attention_mask = torch.tensor([[0, 1.0], [1.0, 0]])

In [ ]:
x.shape

In [ ]:
# import copy
# attention_mask = copy.deepcopy(x)

In [ ]:
# model(inp,attn_mask=None)

In [ ]:
torch.tensor([[2.0, 3]] * 3) + torch.tensor([[2, 3]])

In [ ]:
# class LinearModules(nn.Module):
#     def __init__(self, dim,hidden,heads):
#         super().__init__()
#         self.linear_list = nn.ModuleList([
#            AttentionHead(dim,hidden) for _ in range(heads)
#         ])
#         #print(self.linear_list[0].count_params())
#
#     def forward(self,x,attn_mask=None):
#         print(f"x={x}length={len(x)} in Multi Attention Head\n\n------------")
#         for index,layer in enumerate(self.linear_list):
#             x=self.linear_list[index](x)
#             W0 = layer(x)
#             #print(f"W0={W0}")
#
#             #x=list(x)
#             print(f"x={x}len={len(x)},{type(x)}")
#
#         print(f"x={x}len={len(x)} in Multi Attention Head after")
#         print(f"Concated Heads: {torch.concat(x)}")
#         return torch.concat(x)@W0
#         #return x

In [ ]:
# model = LinearModules(2,3,1)
# model

In [ ]:
# for p in model.parameters():
#     print(p.shape)

In [ ]:
for i, p in model.named_modules():
    print(i, p)

In [ ]:
m = AttentionHead(2, 3)
m.train()
pred = m(torch.tensor([[2.0, 2]] * 3))

In [ ]:
x, y = pred
x

In [ ]:
print(y)

In [ ]:
# model.train()
# pred = model(torch.tensor([[2.0,2]]*3))
# pred

In [ ]:
# list(model.get_submodule("linear_list.0").parameters())

## Part 2.B Multi Attention Head


This line creates a standard linear layer (W_o) that merges the independent representations learned by your individual attention heads back into a single unified vector space.
Here is exactly why it is configured with those dimensions:


#### 1. Fixing the Dimension Explosion


Each individual AttentionHead outputs a tensor of shape (B, T, n_hidden).
When you loop through all your heads and concatenate (torch.cat) them along the final axis, their dimensions combine like this:
$$\text{Shape: } (B, T, \color{lightgreen}{n\_hidden}) \times \text{num\_heads} \xrightarrow{\text{torch.cat}} (B, T, \color{lightblue}{num\_heads \times n\_hidden})$$
If you did not use this projection layer, the data moving to the next block in your neural network would be far too wide (num_heads * n_hidden) instead of the standard model dimension (dim).


#### 2. Allowing Heads to Communicate

Concatenation simply glues the heads next to each other, meaning the features found by Head #1 do not interact with Head #2. Passing the concatenated tensor through nn.Linear performs a matrix multiplication that mathematically mixes all the heads' outputs together.


#### 3. Re-aligning with Residual Connections


In transformer architectures, you usually add the original input $x$ back to the attention output (a residual connection):
$$\text{Output} = x + \text{Attention}(x)$$
For this addition ($+$) to work, the final output must have the exact same shape as the input tensor $x$, which is (B, T, dim). The out_proj layer guarantees this shape matches perfectly.



In [ ]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        # dim: the dimension of the input
        # n_hidden: the hidden dimensions for the attention layer
        # num_heads: the number of attention heads
        super().__init__()
        self.H = num_heads
        # TODO: set up your parameters for multi-head attention. You should initialize
        #       num_heads attention heads (see nn.ModuleList) as well as a linear layer
        #       that projects the concatenated outputs of each head into dim
        #       (what size should this linear layer be?)

        # ======= Answer START ========
        # self.attention = AttentionHead(dim=dim,n_hidden=n_hidden)
        self.linear_list = nn.ModuleList(
            [AttentionHead(dim, n_hidden) for _ in range(num_heads)]
        )
        self.dim = dim

        # 2. Final linear projection layer
        # Maps concatenated outputs (num_heads * n_hidden) back to original dim
        self.W0 = nn.Linear(n_hidden * num_heads, dim)

        # ======= Answer  END ========

    # todo in parallel
    # def forward(
    #     self, x: torch.Tensor, attn_mask: Optional[torch.Tensor]
    # ) -> Tuple[torch.Tensor, torch.Tensor]:
    #     # x                the inputs. shape: (B x T x dim)
    #     # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
    #     #                  contains 1 if (in batch b) token i should attend on token j and 0
    #     #                  otherwise. shape: (B x T x T)
    #     #
    #     # Outputs:
    #     # attn_output      the output of performing multi-headed self-attention on x.
    #     #                  shape: (B x T x dim)
    #     # attn_alphas      the attention weights of each of the attention heads.
    #     #                  shape: (B x Num_heads x T x T)
    #
    #     A,Z,W0 = None, None,None
    #
    #     # TODO: Compute multi-headed attention. Loop through each of your attention heads
    #     #       and collect the outputs. Concatenate them together along the hidden dimension,
    #     #       and then project them back into the output dimension (dim). Return both
    #     #       the final attention outputs as well as the alphas from each head.
    #
    #     # ======= Answer START ========
    #     print(f"x={x}length={len(x)},type={type(x)} in Multi Attention Head\n\n------------")

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        Z_concat = []
        A_concat = []
        head_outputs = []
        head_alphas = []
        print(f"\n\n In Multi headed Attention: {x.shape=}")
        for head in self.linear_list:
            #     Z,A = head(x,attn_mask=attn_mask)
            #     Z_concat.append(Z)
            #     A_concat.append(A)
            # # 4. Concatenate outputs from all heads along the last (hidden) dimension
            # # Resulting shape: (B, T, num_heads * n_hidden)
            # A=torch.cat(A_concat,dim=-1)
            # Z=self.W0(A)
            # Ah=torch.stack(Z_concat,dim=1)
            # return Z,Ah
            out, alpha = head(x, attn_mask=attn_mask)
            print(f"Multi Headed Attention: {out.shape=},{alpha.shape=}")
            print(f"{head=} {out=}")
            head_outputs.append(out)  # Each item shape: (B, T, n_hidden)
            head_alphas.append(alpha)  # Each item shape: (B, T, T)

        # 4. Concatenate outputs from all heads along the last (hidden) dimension
        # Resulting shape: (B, T, num_heads * n_hidden)
        # print(f"{len(head_alphas[0])},{len(head_outputs[0])}")
        concatenated_heads = torch.cat(head_outputs, dim=-1)

        # concatenated_heads=concatenated_heads.transpose(0,1)

        # 5. Project the concatenated representation back to 'dim'
        # Resulting shape: (B, T, dim)
        # print(f"Multi Headed Attention: {concatenated_heads.shape=}")
        attn_output = self.W0(concatenated_heads)
        # print(
        #     f"Multi Headed Attention: {attn_output.shape=} With output head successful\n\n-----------"
        # )

        # 6. Stack attention weights from all heads along a new head dimension
        # Resulting shape: (B, num_heads, T, T)

        attn_alphas = torch.stack(head_alphas, dim=1)
        print(f"{attn_alphas.shape=}")
        print("Multi Headed Attention Successful\n\n\n--------------------------")
        return attn_output, attn_alphas

## Multi Head Attention Parallel

In [ ]:
# todo
class MultiHeadedAttentionParallel(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.Wq = nn.Linear(dim, n_hidden)
        self.Wk = nn.Linear(dim, n_hidden)
        self.Wv = nn.Linear(dim, n_hidden)

        self.H = num_heads
        self.d_h = int(dim / num_heads)
        self.W0 = nn.Linear(n_hidden * num_heads, self.d_h)
        self.layer_size = n_hidden

    def count_params(self):
        return sum(p.view(-1).shape[0] for p in self.parameters())

    def multi_head_qkv(self, x, head):
        Qh = self.Wq(x)
        Kh = self.Wk(x)
        Vh = self.Wv(x)

    def forward(self, x, attn_mask=None):
        Q = self.Wq(x)

In [ ]:
import torch
import torch.nn as nn


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden

        # 1. One combined projection layer for all heads at once
        # This replaces your nn.ModuleList loop completely
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)

        # 2. Final output projection (W0)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape  # Batch Size, Token Size, model_dim?

        # 3. Project all inputs for all heads in parallel
        # Shape becomes: (B, T, num_heads * n_hidden * 3)
        qkv = self.qkv_projection(x)

        # 4. Reshape and split into separate Query, Key, and Value tensors
        # New shape format: (B, num_heads, T, n_hidden)
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)

        # 5. Compute parallel attention weights (Alphas)
        # Shape: (B, num_heads, T, T)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.n_hidden**0.5)

        if attn_mask is not None:
            # Mask format: Convert 0s to a very large negative value (-1e9)
            # Unsqueeze mask to match (B, 1, T, T) for broadcasting across heads
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))

        attn_alphas = torch.softmax(scores, dim=-1)

        # 6. Apply weights to values
        # Shape: (B, num_heads, T, n_hidden)
        context = attn_alphas @ v

        # 7. Concatenate all heads back together efficiently
        # Permute back to (B, T, num_heads, n_hidden) -> Flatten to (B, T, num_heads * n_hidden)
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        # 8. Run through final output projection W0
        Z = self.W0(A)

        return Z, attn_alphas

### With dropout

In [ ]:
class AttentionHead(nn.Module):
    """
    Multi Attention Head
    Splits Input in Q,K,V

    """

    def __init__(self, model_dim, H, dropout_rate=0.1):
        super().__init__()
        self.Wq = nn.Linear(model_dim, model_dim)
        self.Wk = nn.Linear(model_dim, model_dim)
        self.Wv = nn.Linear(model_dim, model_dim)
        self.H = H
        self.d_h = int(model_dim / H)
        self.dropout = nn.Dropout(p=dropout_rate)

        self.Wo = nn.Linear(model_dim, model_dim)

    def forward(self, sequences, attn_mask=False):
        """Input shape: [batch_size, seq_len, d_model=num_head * d_head]
        # if key_value_states are provided this layer is used as a cross-attention layer for text translation..

        # for the decoder

        """
        batch_size, seq_len, model_dim = sequences.size()
        Q = self.Wq(sequences)

        K, V = self.Wk(sequences), self.Wv(sequences)

        A = Q @ K.transpose(-2, -1)
        if attn_mask is not None and attn_mask:
            A = A.masked_fill(attn_mask == 0, -float("inf"))
        A = F.softmax(A / self.d_h**0.5, dim=-1)  # Applying softmax
        A = self.dropout(A)  # Final

        # Output Z

        Z = A @ V  # torch,tensor
        print(f"{Z.shape=}")
        ### Concatenating in parallel along heads and sequences
        ## Because continuous input
        # Z = (Z.contiguous().view(
        #     batch_size,seq_len,self.H*self.d_h)
        # )
        # 2. Transpose to move seq_len before H: shape (batch_size, seq_len, H, d_h)
        # NOTE: Z is now NON-CONTIGUOUS in memory!
        Z = Z.transpose(1, 2).reshape(batch_size, seq_len, model_dim)
        # 3. Concatenate all heads into d_model = H * d_h
        # Fails without .contiguous():
        # Z = Z.contiguous().view(batch_size,seq_len,self.H*self.d_h)
        # final linear projections
        Z = self.Wo(Z)
        return A, Z

In [ ]:
a = torch.tensor([[1, 2], [2, 3]])
a

In [ ]:
a.unsqueeze(dim=0).shape

In [ ]:
a.unsqueeze(dim=1).shape

In [ ]:
class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        # dim: the dimension of the input
        # n_hidden: the hidden dimensions for the attention layer
        # num_heads: the number of attention heads
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        # TODO: set up your parameters for multi-head attention. You should initialize
        #       num_heads attention heads (see nn.ModuleList) as well as a linear layer
        #       that projects the concatenated outputs of each head into dim
        #       (what size should this linear layer be?)

        # self.Wq = nn.Linear(dim,n_hidden)
        # self.Wk = nn.Linear(dim,n_hidden)
        # self.Wv = nn.Linear(dim,n_hidden)

        self.qkv = nn.Linear(
            dim, num_heads * n_hidden * 3, bias=False
        )  # Why bias false?

        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        print(f"{QKV.shape=}")
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        print(f"After reshaping reshaped: {QKV.shape=}")
        QKV = QKV.transpose(-2, -1)
        Q, K, V = QKV.chunk(3, dim=-1)

        # First rescaling then applying attention mask if given
        A = Q @ K.transpose(-2, -1) / (self.n_hidden**0.5)

        scores = A

        if attn_mask is not None:
            """
            causal masking future inputs of 0 to negative so that softmax will return 0
            """
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        # Normalizing inputs across each layer
        A = torch.softmax(scores, dim=-1)

        # Calculating output for 1 head

        Z = A @ V

        # Concatenating outputs transpose and reshape along heads
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)

        # Final Projection
        Z = self.WO(Z)

        # return output, attention
        return Z, A

In [ ]:
print(torch.stack((Z, Z), dim=1).shape)
print(torch.stack((Z, Z), dim=-1).shape)

In [ ]:
print(torch.cat([Z, Z], dim=0).shape)

print(torch.cat([Z, Z], dim=-1).shape)
print(torch.cat([Z, Z], dim=1).shape)

In [ ]:
print(torch.concat([Z, Z], dim=0).shape)

print(torch.concat([Z, Z], dim=1).shape)
print(torch.concat([Z, Z], dim=-1).shape)

In [ ]:
torch.manual_seed(42)
x = torch.randn(10, 3, 3)

In [ ]:
multi_attn_head_model = MultiHeadedAttentionParallel(dim=3, n_hidden=9, num_heads=2)

In [ ]:
multi_attn_head_model

In [ ]:
list(multi_attn_head_model.named_modules())

#### Each head will pay attention to its own set of W,K,V

In [ ]:
Z1, A = multi_attn_head_model(x)

In [ ]:
Z1

In [ ]:
A

In [ ]:
Z1.shape

In [ ]:
A.shape

In [ ]:
A.requires_grad

In [ ]:
# multi_attn_head_model(x,attn_mask=None)

## Part 2.C

### Feed Forward Network

In [ ]:
class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        # dim: the dimension of the input
        # n_hidden: the hidden dimensions for the attention layer
        # num_heads: the number of attention heads
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        # TODO: set up your parameters for multi-head attention. You should initialize
        #       num_heads attention heads (see nn.ModuleList) as well as a linear layer
        #       that projects the concatenated outputs of each head into dim
        #       (what size should this linear layer be?)

        # self.Wq = nn.Linear(dim,n_hidden)
        # self.Wk = nn.Linear(dim,n_hidden)
        # self.Wv = nn.Linear(dim,n_hidden)

        self.qkv = nn.Linear(
            dim, num_heads * n_hidden * 3, bias=False
        )  # Why bias false?

        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        print(f"{QKV.shape=}")
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        print(f"After reshaping reshaped: {QKV.shape=}")
        QKV = QKV.transpose(-2, -1)
        Q, K, V = QKV.chunk(3, dim=-1)

        # First rescaling then applying attention mask if given
        A = Q @ K.transpose(-2, -1) / (self.n_hidden**0.5)

        scores = A

        if attn_mask is not None:
            """
            causal masking future inputs of 0 to negative so that softmax will return 0
            """
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        # Normalizing inputs across each layer
        A = torch.softmax(scores, dim=-1)

        # Calculating output for 1 head

        Z = A @ V

        # Concatenating outputs transpose and reshape along heads
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)

        # Final Projection
        Z = self.WO(Z)

        # return output, attention
        return Z, A


import torch
import torch.nn as nn


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden

        # 1. One combined projection layer for all heads at once
        # This replaces your nn.ModuleList loop completely
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)

        # 2. Final output projection (W0)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape  # Batch Size, Token Size, model_dim?

        # 3. Project all inputs for all heads in parallel
        # Shape becomes: (B, T, num_heads * n_hidden * 3)
        qkv = self.qkv_projection(x)

        # 4. Reshape and split into separate Query, Key, and Value tensors
        # New shape format: (B, num_heads, T, n_hidden)
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)

        # 5. Compute parallel attention weights (Alphas)
        # Shape: (B, num_heads, T, T)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.n_hidden**0.5)

        if attn_mask is not None:
            # Mask format: Convert 0s to a very large negative value (-1e9)
            # Unsqueeze mask to match (B, 1, T, T) for broadcasting across heads
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))

        attn_alphas = torch.softmax(scores, dim=-1)

        # 6. Apply weights to values
        # Shape: (B, num_heads, T, n_hidden)
        context = attn_alphas @ v

        # 7. Concatenate all heads back together efficiently
        # Permute back to (B, T, num_heads, n_hidden) -> Flatten to (B, T, num_heads * n_hidden)
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        # 8. Run through final output projection W0
        Z = self.W0(A)

        return Z, attn_alphas

In [ ]:
class FFN(nn.Module):
    def __init__(self, dim: int, n_hidden: int):
        # dim       the dimension of the input
        # n_hidden  the width of the linear layer

        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, n_hidden),
            nn.GELU(),
            nn.Linear(n_hidden, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x         the input. shape: (B x T x dim)

        # Outputs:
        # out       the output of the feed-forward network: (B x T x dim)
        # print("\n\n\nIn FFN-------------------------")
        print(f"{x.shape=}")
        out = self.net(x)
        # print(f"Out={out.shape=}")
        return out

In [ ]:
def print_variance(name, data):
    # First dimension(rows) is batch elements
    # Second dimension(columns) is neurons.
    # np_data = data.detach().numpy()

    # Compute variance across neurons and average these variances over members of the batch
    neuron_variance = torch.mean(torch.var(data, dim=0))
    # Print out the name and the variance
    print(f"{name=}, Variance={neuron_variance.float()}")

### Attention Residual

In [ ]:
import torch.nn.functional as F

In [ ]:
device = "mps"

In [ ]:
MultiHeadedAttention

In [ ]:
# these are already implemented for you!
def print_variance(name, data):
    # First dimension(rows) is batch elements
    # Second dimension(columns) is neurons.
    # np_data = data.detach().numpy()

    # Compute variance across neurons and average these variances over members of the batch
    neuron_variance = torch.mean(torch.var(data, dim=0))
    # Print out the name and the variance
    print(f"{name=}, Variance={neuron_variance.float()}")


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        super().__init__()

        # self.attn = MultiAttentionHead(dim, attn_dim, num_heads)
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            # nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            # nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, attn_mask=False
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
        #                  contains 1 if (in batch b) token i should attend on token j and 0
        #                  otherwise. shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      the attention weights of each of the attention heads.
        #                  shape: (B x Num_heads x T x T)
        # print("\n\n--------- Attention Residual")
        print_variance("Input", x)
        Z, A = self.attn(x=self.norm1(x), attn_mask=attn_mask)
        # Z,A = attn_out,alphas
        print_variance("after first attention block output", Z)

        # print("\n\n  In Attention Residual")
        # print(f"{Z.shape=}\t,{x.shape=}\t.{A.shape=}\n")
        x = Z + x

        print_variance("after residual 1 adding output to Z", x)

        ffn_out = self.ffn(self.norm2(x))
        x = ffn_out + x

        print_variance("after Applying FeedForward output with residual", x)
        # print_variance("Attention Variance:",A)
        print("Residual Block\n\n")
        # print(f"{x.shape=}\n\n--------------")
        return x, A

### Debug

In [ ]:
MultiHeadedAttention(1, 1, 1)

In [ ]:
input_features = 64
output_features = 64
dim = 3

In [ ]:
print(x.shape)
model_ffn = nn.Sequential(
    nn.LayerNorm(x.shape[-1], eps=1e-6),
    nn.Linear(64, 1000),
    nn.ReLU(),
    nn.Linear(1000, output_features),
).to(device)
layer_norm = nn.LayerNorm(x.shape[-1]).to(device)
out = layer_norm(x)

In [ ]:
out[0, :, :]

In [ ]:
F.gelu(x)

In [ ]:
model_ffn

In [ ]:
x = torch.randn(input_features, dim, input_features)

In [ ]:
model_ffn(x).shape

In [ ]:
# for key, value in model_ffn.state_dict().items():
#     print(key, value.shape)

In [ ]:
print(list(model_ffn.named_modules()))

In [ ]:
device = "mps"

In [ ]:
num_tokens = 100
batch_size = 10
dim = 3
num_layers = 4
num_heads = 2
attn_dim = 32
model = AttentionResidual(
    dim=dim, attn_dim=attn_dim, mlp_dim=dim, num_heads=num_heads
).to(device)

In [ ]:
model

In [ ]:
print(list(model.named_modules()))

In [ ]:
# (Batch Size = 32, Sequence Length = 3, Dim = 3)
x = torch.randn(batch_size, num_tokens, dim).to(device)

out, A = model(x, attn_mask=None)
print(out.shape, A.shape)

In [ ]:
x

In [ ]:
model

In [ ]:
print(list(model.named_modules()))

In [ ]:
# model(x,attn_mask=None)

## Transformer
 The FFN performs
normalization along with linear layers interspersed by GELUs 2 (similar to the MLPs
you’ve seen in previous PSETs). AttentionResidual then sends the input through
both multiheaded attention and the FFN, adding a residual after each step.
Deliverable Implement Transformer, which passes the input through successive
AttentionResidual layers.

In [ ]:
import torch
from torch import nn

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

In [ ]:
# device = ["cuda","mps","cpu"]

In [ ]:
# a utility for calculating running average
class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


def averager(func):
    num = 0
    total = 0

    # val = 0.0

    def update(val, sz):
        nonlocal num
        nonlocal total
        num += val * sz
        total += sz

    @wraps(func)
    def inner():
        nonlocal num, total
        update(num / total)

    return inner

In [ ]:
def half(a, b):
    return a, b

In [ ]:
avg_meter = AverageMeter()
avg_meter.update(2, 3)

In [ ]:
# #@averager
# def gcd(a,b):
#     if a==0:
#         return b
#     else:
#         return gcd(a/b,a)

In [ ]:
# gcd(2,3)

In [ ]:
qkv_projection = nn.Linear(in_features=10, out_features=3 * 3 * 3, bias=False)

In [ ]:
x = torch.randn(27, 10)
qkv = qkv_projection(x)

In [ ]:
qkv.shape

In [ ]:
q, k, v = qkv.view(3, 27, 9)

In [ ]:
type(q)

In [ ]:
q.shape

In [ ]:
len(qkv.chunk(3, dim=-2))

In [ ]:
# these are already implemented for you!


def print_variance(name, data):
    # First dimension(rows) is batch elements
    # Second dimension(columns) is neurons.
    # np_data = data.detach().numpy()

    # Compute variance across neurons and average these variances over members of the batch
    neuron_variance = torch.mean(torch.var(data, dim=0))
    # Print out the name and the variance
    print(f"{name=}, Variance={neuron_variance.float()}")


class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        # dim: the dimension of the input
        # n_hidden: the hidden dimensions for the attention layer
        # num_heads: the number of attention heads
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        # TODO: set up your parameters for multi-head attention. You should initialize
        #       num_heads attention heads (see nn.ModuleList) as well as a linear layer
        #       that projects the concatenated outputs of each head into dim
        #       (what size should this linear layer be?)

        # self.Wq = nn.Linear(dim,n_hidden)
        # self.Wk = nn.Linear(dim,n_hidden)
        # self.Wv = nn.Linear(dim,n_hidden)

        self.qkv = nn.Linear(
            dim, num_heads * n_hidden * 3, bias=False
        )  # Why bias false?

        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        # print(f"{QKV.shape=}")
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        # print(f"After reshaping reshaped: {QKV.shape=}")
        QKV = QKV.transpose(-2, -1)
        Q, K, V = QKV.chunk(3, dim=-1)

        # First rescaling then applying attention mask if given
        A = Q @ K.transpose(-2, -1) / (self.n_hidden**0.5)

        scores = A

        if attn_mask is not None:
            """
            causal masking future inputs of 0 to negative so that softmax will return 0
            """
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        # Normalizing inputs across each layer
        A = torch.softmax(scores, dim=-1)

        # Calculating output for 1 head

        Z = A @ V

        # Concatenating outputs transpose and reshape along heads
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)

        # Final Projection
        Z = self.WO(Z)

        # return output, attention
        return Z, A


import torch
import torch.nn as nn


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden

        # 1. One combined projection layer for all heads at once
        # This replaces your nn.ModuleList loop completely
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)

        # 2. Final output projection (W0)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape  # Batch Size, Token Size, model_dim?

        # 3. Project all inputs for all heads in parallel
        # Shape becomes: (B, T, num_heads * n_hidden * 3)
        qkv = self.qkv_projection(x)

        # 4. Reshape and split into separate Query, Key, and Value tensors
        # New shape format: (B, num_heads, T, n_hidden)
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)

        # 5. Compute parallel attention weights (Alphas)
        # Shape: (B, num_heads, T, T)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.n_hidden**0.5)

        if attn_mask is not None:
            # Mask format: Convert 0s to a very large negative value (-1e9)
            # Unsqueeze mask to match (B, 1, T, T) for broadcasting across heads
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))

        attn_alphas = torch.softmax(scores, dim=-1)

        # 6. Apply weights to values
        # Shape: (B, num_heads, T, n_hidden)
        context = attn_alphas @ v

        # 7. Concatenate all heads back together efficiently
        # Permute back to (B, T, num_heads, n_hidden) -> Flatten to (B, T, num_heads * n_hidden)
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        # 8. Run through final output projection W0
        Z = self.W0(A)

        return Z, attn_alphas


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        super().__init__()

        # self.attn = MultiAttentionHead(dim, attn_dim, num_heads)
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            # nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, attn_mask=False
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
        #                  contains 1 if (in batch b) token i should attend on token j and 0
        #                  otherwise. shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      the attention weights of each of the attention heads.
        #                  shape: (B x Num_heads x T x T)
        # print("\n\n--------- Attention Residual")
        print_variance("Input", x)
        Z, A = self.attn(x=self.norm1(x), attn_mask=attn_mask)
        # Z,A = attn_out,alphas
        print_variance("after first attention block output", Z)

        # print("\n\n  In Attention Residual")
        # print(f"{Z.shape=}\t,{x.shape=}\t.{A.shape=}\n")
        x = Z + x

        print_variance("after residual 1 adding output to Z", x)

        # ffn_out = F.gelu(self.ffn(self.norm2(x)))
        ffn_out = self.ffn(self.norm2(x))
        x = ffn_out + x

        print_variance("after Applying FeedForward output with residual", x)
        # print_variance("Attention Variance:",A)
        print("Residual Block\n\n")
        # print(f"{x.shape=}\n\n--------------")
        return x, A


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        # num_layers the number of attention layers.
        super().__init__()

        # ======= Answer START ========
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

        # ======= Answer END ========

    def forward(
        self, x: torch.Tensor, attn_mask=False, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []
        Z = None
        A = []
        # TODO: Implement the transformer forward pass! Pass the input successively through each of the AttentionResidual layers. If return_attn is True, collect the alphas along the way.

        # ======= Answer START ========
        print(f"\n\n-------------------------Transformer: {x.shape}")
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)

            # if return_attn is None:
            #     alphas = None
            # print(type(alphas), alphas.shape)
            if return_attn is not None:
                A.append(alphas)

        # print(x.shape)
        # print("return attention:", return_attn)
        # ======= Answer END ========
        if return_attn:
            return x, torch.stack(A, dim=1)
        else:
            return x, None

        # return x,None if   return_attn is False  else x, torch.stack(A,dim=1)
        # return output, collected_attns

In [ ]:
class Transformer2(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        # num_layers the number of attention layers.
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadedAttentionParallel(dim, attn_dim, num_heads)

        self.ffn = [
            nn.ModuleList(
                nn.Sequential(
                    nn.Linear(dim, attn_dim), nn.GELU(), nn.Linear(attn_dim, dim)
                )
                for _ in range(num_layers)
            )
        ]
        self.norm2 = nn.LayerNorm(dim)

        # ======= Answer END ========

    def forward(
        self, x: torch.Tensor, attn_mask=None, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []

        # TODO: Implement the transformer forward pass! Pass the input successively through each of the
        # AttentionResidual layers. If return_attn is True, collect the alphas along the way.

        # ======= Answer START ========
        print(f"\n\n-------------------------Transformer: {x.shape}")
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)
            # if return_attn is None:
            #     alphas = None
            # print(type(alphas), alphas.shape)
            if return_attn is not None:
                collected_attns.append(alphas)

        # print(x.shape)
        # print("return attention:", return_attn)
        # ======= Answer END ========
        if return_attn is False:
            return x, None
        else:
            return (
                x,
                torch.stack(collected_attns, dim=1),
            )
        # return x if not return_attn else x,collected_attns
        # return output, collected_attns

In [ ]:
device = "mps"

### Debug

In [ ]:
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
attn_dim = 32
inp = torch.randn(batch_size, num_tokens, dim).to(device)
model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)

In [ ]:
out, a = model(inp, attn_mask=None, return_attn=True)

In [ ]:
out

Test your transformer implementation here

In [ ]:
def test_transformer():
    num_tokens = 100
    batch_size = 10
    dim = 64
    num_layers = 4
    num_heads = 5
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    )

    inp = torch.randn(batch_size, num_tokens, dim)
    print(f"Input: {inp.shape=}")
    # test case 1 regular forward pass
    print("Test Case 1")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None)
        # assert alpha is None
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"
    print("Passed")
    # test case 2 collect attentions
    print("Test Case 2")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None, return_attn=True)
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"
        print("Test case 2 output correct")
        assert alpha.shape == (
            batch_size,
            num_layers,
            num_heads,
            num_tokens,
            num_tokens,
        ), f"wrong alpha shape {alpha.shape}"

    print("Test Case 3")
    # test case 3 with attention mask
    attn_mask = torch.zeros(batch_size, num_tokens, num_tokens)
    attn_mask[:, torch.arange(num_tokens), torch.arange(num_tokens)] = 1
    attn_mask[:, torch.arange(num_tokens)[1:], torch.arange(num_tokens)[:-1]] = 1
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=attn_mask, return_attn=True)
        print("Attention mask pattern", attn_mask[0])
        print("Alpha pattern", alpha[0, 0, 0])
        assert torch.all(alpha.permute(1, 2, 0, 3, 4)[:, :, attn_mask == 0] == 0).item()
    print("Test case 3 passed!\n\n")
    print("Test Case 4")
    # test case 4 creates a causal mask where each token can only attend to previous tokens and itself
    causal_mask = (
        torch.tril(torch.ones(num_tokens, num_tokens))
        .unsqueeze(0)
        .repeat(batch_size, 1, 1)
    )  # Shape: (B, T, T)

    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=causal_mask, return_attn=True)
        # Verify the causal mask
        for b in range(batch_size):
            for l in range(num_layers):
                for h in range(num_heads):
                    attn_weights = alpha[b, l, h]  # Shape: (T, T)
                    # Positions where j > i should have zero attention weights
                    # We can create a boolean mask for j > i
                    future_mask = torch.triu(
                        torch.ones(num_tokens, num_tokens), diagonal=1
                    ).bool()  # Shape: (T, T)
                    # Extract attention weights for future positions
                    future_attn = attn_weights[future_mask]
                    # Assert that these weights are close to zero
                    assert torch.all(
                        future_attn < 1e-6
                    ), f"Causal mask violated in batch {b}, layer {l}, head {h}"

    torch.save(dummy_model, "transformer_mask.pth")


test_transformer()

### Debug

In [ ]:
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
attn_dim = 32
dummy_model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)

inp = torch.randn(batch_size, num_tokens, dim).to(device)
print(f"Input: {inp.shape=}")
# test case 1 regular forward pass
print("Test Case 1")
with torch.no_grad():
    output, alpha = dummy_model(inp, attn_mask=None)
    if len(output) == 0:
        output = None
torch.save(dummy_model, "model.pth")

In [ ]:
alpha

In [ ]:
output.shape

## Test Transformer

In [ ]:
def test_transformer():
    num_tokens = 100
    batch_size = 10
    dim = 64
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    ).to(device)

    inp = torch.randn(batch_size, num_tokens, dim).to(device)
    print(f"Input: {inp.shape=}")
    # test case 1 regular forward pass
    print("Test Case 1")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None)
        # print(f"\n\n\n{output.shape=},{alpha.shape=}")

        # if len(alpha)==0:
        #     alpha=None
        assert alpha is None
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"

    # test case 2 collect attentions


test_transformer()

In [ ]:
def test_transformer():
    num_tokens = 100
    batch_size = 10
    dim = 64
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    ).to(device)

    inp = torch.randn(batch_size, num_tokens, dim).to(device)
    print(f"Input: {inp.shape=}")
    # test case 1 regular forward pass
    print("Test Case 1")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None)
        # print(f"\n\n\n{output.shape=},{alpha.shape=}")

        # if len(alpha)==0:
        #     alpha=None
        assert alpha is None
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"

    # test case 2 collect attentions


test_transformer()

In [ ]:
def test_transformer():
    num_tokens = 100
    batch_size = 10
    dim = 64
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    ).to(device)

    inp = torch.randn(batch_size, num_tokens, dim).to(device)
    print(f"Input: {inp.shape=}")
    # test case 1 regular forward pass
    print("Test Case 1")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None)
        # print(f"\n\n\n{output.shape=},{alpha.shape=}")

        # if len(alpha)==0:
        #     alpha=None
        assert alpha is None
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"

    # test case 2 collect attentions


test_transformer()

In [ ]:
def test_transformer_2():
    num_tokens = 100
    batch_size = 10
    dim = 64
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    ).to(device)

    inp = torch.randn(batch_size, num_tokens, dim).to(device)
    print(f"Input: {inp.shape=}")
    print("Test Case 2")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None, return_attn=True)
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"
        assert alpha.shape == (
            batch_size,
            num_layers,
            num_heads,
            num_tokens,
            num_tokens,
        ), f"wrong alpha shape {alpha.shape}"

    print("Test Case 3")
    # test case 3 with attention mask
    attn_mask = torch.zeros(batch_size, num_tokens, num_tokens).to(device)
    attn_mask[:, torch.arange(num_tokens), torch.arange(num_tokens)] = 1
    attn_mask[:, torch.arange(num_tokens)[1:], torch.arange(num_tokens)[:-1]] = 1
    attn_mask = attn_mask.to(device)
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=attn_mask, return_attn=True)
        print("Attention mask pattern", attn_mask[0])
        print("Alpha pattern", alpha[0, 0, 0])
        assert torch.all(alpha.permute(1, 2, 0, 3, 4)[:, :, attn_mask == 0] == 0).item()

    print("Test Case 4")
    # test case 4 creates a causal mask where each token can only attend to previous tokens and itself
    causal_mask = (
        torch.tril(torch.ones(num_tokens, num_tokens))
        .unsqueeze(0)
        .repeat(batch_size, 1, 1)
    ).to(
        device
    )  # Shape: (B, T, T)

    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=causal_mask, return_attn=True)
        # Verify the causal mask
        i = 0
        future_mask = None
        for b in range(batch_size):
            for l in range(num_layers):
                for h in range(num_heads):
                    attn_weights = alpha[b, l, h]  # Shape: (T, T)
                    # Positions where j > i should have zero attention weights
                    # We can create a boolean mask for j > i
                    future_mask = torch.triu(
                        torch.ones(num_tokens, num_tokens), diagonal=1
                    ).bool()  # Shape: (T, T)
                    # if i==0:
                    #     print(future_mask)
                    # Extract attention weights for future positions
                    future_attn = attn_weights[future_mask]
                    # Assert that these weights are close to zero
                    assert torch.all(
                        future_attn < 1e-6
                    ), f"Causal mask violated in batch {b}, layer {l}, head {h}"

        print(future_mask)


test_transformer_2()

In [ ]:
def test_transformer_2():
    num_tokens = 1
    batch_size = 1
    dim = 2
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    ).to(device)

    inp = torch.randn(batch_size, num_tokens, dim).to(device)
    print(f"Input: {inp.shape=}")
    print("Test Case 2")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None, return_attn=True)
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"
        assert alpha.shape == (
            batch_size,
            num_layers,
            num_heads,
            num_tokens,
            num_tokens,
        ), f"wrong alpha shape {alpha.shape}"

    print("Test Case 3")
    # test case 3 with attention mask
    attn_mask = torch.zeros(batch_size, num_tokens, num_tokens).to(device)
    attn_mask[:, torch.arange(num_tokens), torch.arange(num_tokens)] = 1
    attn_mask[:, torch.arange(num_tokens)[1:], torch.arange(num_tokens)[:-1]] = 1
    attn_mask = attn_mask.to(device)
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=attn_mask, return_attn=True)
        print("Attention mask pattern", attn_mask[0])
        print("Alpha pattern", alpha[0, 0, 0])
        assert torch.all(alpha.permute(1, 2, 0, 3, 4)[:, :, attn_mask == 0] == 0).item()

    print("Test Case 4")
    # test case 4 creates a causal mask where each token can only attend to previous tokens and itself
    causal_mask = (
        torch.tril(torch.ones(num_tokens, num_tokens))
        .unsqueeze(0)
        .repeat(batch_size, 1, 1)
    ).to(
        device
    )  # Shape: (B, T, T)

    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=causal_mask, return_attn=True)
        # Verify the causal mask
        i = 0
        future_mask = None
        for b in range(batch_size):
            for l in range(num_layers):
                for h in range(num_heads):
                    attn_weights = alpha[b, l, h]  # Shape: (T, T)
                    # Positions where j > i should have zero attention weights
                    # We can create a boolean mask for j > i
                    future_mask = torch.triu(
                        torch.ones(num_tokens, num_tokens), diagonal=1
                    ).bool()  # Shape: (T, T)
                    # if i==0:
                    #     print(future_mask)
                    # Extract attention weights for future positions
                    future_attn = attn_weights[future_mask]
                    # Assert that these weights are close to zero
                    assert torch.all(
                        future_attn < 1e-6
                    ), f"Causal mask violated in batch {b}, layer {l}, head {h}"

        print(future_mask)


test_transformer_2()

In [ ]:
def test_transformer_2():
    num_tokens = 1
    batch_size = 1
    dim = 2
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    ).to(device)

    inp = torch.randn(batch_size, num_tokens, dim).to(device)
    print(f"Input: {inp.shape=}")
    print("Test Case 2")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None, return_attn=True)
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"
        assert alpha.shape == (
            batch_size,
            num_layers,
            num_heads,
            num_tokens,
            num_tokens,
        ), f"wrong alpha shape {alpha.shape}"

    print("Test Case 3")
    # test case 3 with attention mask
    attn_mask = torch.zeros(batch_size, num_tokens, num_tokens).to(device)
    attn_mask[:, torch.arange(num_tokens), torch.arange(num_tokens)] = 1
    attn_mask[:, torch.arange(num_tokens)[1:], torch.arange(num_tokens)[:-1]] = 1
    attn_mask = attn_mask.to(device)
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=attn_mask, return_attn=True)
        print("Attention mask pattern", attn_mask[0])
        print("Alpha pattern", alpha[0, 0, 0])
        assert torch.all(alpha.permute(1, 2, 0, 3, 4)[:, :, attn_mask == 0] == 0).item()

    print("Test Case 4")
    # test case 4 creates a causal mask where each token can only attend to previous tokens and itself
    causal_mask = (
        torch.tril(torch.ones(num_tokens, num_tokens))
        .unsqueeze(0)
        .repeat(batch_size, 1, 1)
    ).to(
        device
    )  # Shape: (B, T, T)

    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=causal_mask, return_attn=True)
        # Verify the causal mask
        i = 0
        future_mask = None
        for b in range(batch_size):
            for l in range(num_layers):
                for h in range(num_heads):
                    attn_weights = alpha[b, l, h]  # Shape: (T, T)
                    # Positions where j > i should have zero attention weights
                    # We can create a boolean mask for j > i
                    future_mask = torch.triu(
                        torch.ones(num_tokens, num_tokens), diagonal=1
                    ).bool()  # Shape: (T, T)
                    # if i==0:
                    #     print(future_mask)
                    # Extract attention weights for future positions
                    future_attn = attn_weights[future_mask]
                    # Assert that these weights are close to zero
                    assert torch.all(
                        future_attn < 1e-6
                    ), f"Causal mask violated in batch {b}, layer {l}, head {h}"

        print(future_mask)


test_transformer_2()

In [ ]:
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
dummy_model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)

inp = torch.randn(batch_size, num_tokens, dim).to(device)
print(f"Input: {inp.shape=}")
print("Test Case 2")
with torch.no_grad():
    output, alpha = dummy_model(inp, attn_mask=None, return_attn=True)
    assert output.shape == (
        batch_size,
        num_tokens,
        dim,
    ), f"wrong output shape {output.shape}"
    assert alpha.shape == (
        batch_size,
        num_layers,
        num_heads,
        num_tokens,
        num_tokens,
    ), f"wrong alpha shape {alpha.shape}"

In [ ]:
alpha

In [ ]:
dummy_model = Transformer(dim, attn_dim, dim, 4, 10).to(device)
print("Test Case 4")
# test case 4 creates a causal mask where each token can only attend to previous tokens and itself
causal_mask = (
    torch.tril(torch.ones(num_tokens, num_tokens)).unsqueeze(0).repeat(batch_size, 1, 1)
).to(
    device
)  # Shape: (B, T, T) # masking lower halves 1 upper halves 0

with torch.no_grad():
    output, alpha = dummy_model(inp, attn_mask=causal_mask, return_attn=True)
    # Verify the causal mask
    i = 0
    future_mask = None
    for b in range(batch_size):
        for l in range(num_layers):
            for h in range(num_heads):
                attn_weights = alpha[b, l, h]  # Shape: (T, T)
                # Positions where j > i should have zero attention weights
                # We can create a boolean mask for j > i
                future_mask = torch.triu(
                    torch.ones(num_tokens, num_tokens), diagonal=1
                ).bool()  # Shape: (T, T)
                # if i==0:
                #     print(future_mask)
                # Extract attention weights for future positions
                future_attn = attn_weights[future_mask]
                # Assert that these weights are close to zero
                assert torch.all(
                    future_attn < 1e-6
                ), f"Causal mask violated in batch {b}, layer {l}, head {h}"

    # print(future_mask)
    print(future_attn)

In [ ]:
future_attn.all() == 0

In [ ]:
causal_mask

In [ ]:
future_mask

In [ ]:
scripted_model = torch.jit.script(model)
scripted_model.save("scripted_final.pt")

In [ ]:
torch.save(dummy_model.state_dict(), "final_transformer3.pth")

In [ ]:
print("Test Case 3")
# test case 3 with attention mask
attn_mask = torch.zeros(batch_size, num_tokens, num_tokens).to(device)
attn_mask[:, torch.arange(num_tokens), torch.arange(num_tokens)] = 1
attn_mask[:, torch.arange(num_tokens)[1:], torch.arange(num_tokens)[:-1]] = 1
attn_mask = attn_mask.to(device)
with torch.no_grad():
    output, alpha = dummy_model(inp, attn_mask=attn_mask, return_attn=True)
    print("Attention mask pattern", attn_mask[0])
    print("Alpha pattern", alpha[0, 0, 0])
    assert torch.all(alpha.permute(1, 2, 0, 3, 4)[:, :, attn_mask == 0] == 0).item()

In [ ]:
alpha

### Debug

In [ ]:
w = torch.randn(32, 200, 100)

In [ ]:
20000 / 32

In [ ]:
w.transpose(dim0=0, dim1=1).shape

In [ ]:
Q = torch.randn(32, 100, 10)
K = torch.randn(32, 100, 10)
(Q @ K.transpose(-2, -1)).shape

In [ ]:
Q1 = torch.tensor([[1], [2]])
V1 = copy.deepcopy(Q1)
V1_T = V1.transpose(0, -1)
V1_T

In [ ]:
Q1 @ V1_T

## Tokenization

Transformers are typically designed to handle discrete token sequences, like
words in a sentence. However, in many domains like audio and images, the input is continuous rather than discrete. Explain
how the transformer architecture can be adapted to handle continuous inputs, such
as audio and images. Think about how tokenization, embedding, and positional
encodings need to be adjusted to effectively apply self-attention to these types of
inputs. Consider how you would divide continuous data into meaningful ”tokens”
and how the transformer can capture local and global relationships within this data.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
with open("testing_data/dracula.txt") as f:
    text = f.read()

### Tokenizing one hot

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(text.split(" "))

In [ ]:
X[:10].toarray()

In [ ]:
X = X.T

In [ ]:
X.shape

In [ ]:
X[0].toarray()

In [ ]:
vocab = vectorizer.vocabulary_

In [ ]:
vocab

In [ ]:
from collections import Counter

import torch

# 1. Sample Corpus
corpus = text

# 2. Tokenize (Simple space split)
tokenized_corpus = [text.split() for text in corpus if len(text.split()) > 2]

# 3. Build Vocabulary
token_counts = Counter()
for tokens in tokenized_corpus:
    token_counts.update(tokens)

# Map unique words to integers, keeping 0 for padding
vocab = {word.lower(): i + 1 for i, (word, _) in enumerate(token_counts.items())}
vocab["pad"] = 0  # Special padding token
print("Vocabulary:", vocab)

In [ ]:
for x in tokenized_corpus[:20]:
    print(x)

In [ ]:
tokenized_corpus[:20]

In [ ]:
dummy_model

In [ ]:
# 4. Map text to numerical indices
numerical_sequences = [
    [vocab[token.lower()] for token in tokens] for tokens in tokenized_corpus
]

# 6. Convert to PyTorch Tensor

In [ ]:
max_length = max(len(seq) for seq in numerical_sequences)

In [ ]:
padded_sequence = [
    seq + [vocab["<pad>"]] * (max_length - len(seq)) for seq in numerical_sequences
]

In [ ]:
padded_sequences = padded_sequence

In [ ]:
text_tensor = torch.tensor(padded_sequences, dtype=torch.float32)

In [ ]:
text_tensor[:20]

In [ ]:
text_tensor.shape

## Problem 3: Vision Transformer

## Part 3.A

### given


```python
    # x : input data (RGB image)
    # K : tokenization patch size
    # d : token/query/key/value dimensionality (setting these all as the same)
    # L : number of layers
    # W_q_T, W_k_T, W_v_T : transposed query/key/value projection matrices
    # mlp: tokenwise mlps
    # tokenize input image
    T = tokenize(x,K) # 3 x H x W image --> N x d array of token code vectors
    # run tokens through all L layers
    for l in range(L):
    # attention layer
    Q, K, V = nn.matmul(nn.layernorm(T),[W_q_T[l], W_k_T[l], W_v_T[l]])
    # nn.matmul does matrix multiplication
    A = nn.softmax(nn.matmul(Q,K.transpose()), dim=0)/sqrt(d)
    T = nn.matmul(A,V) + T # note residual connection
    # tokenwise mlp
    T = mlp[l](nn.layernorm(T)) + T # note residual connection
    # T now contains the output token representation computed by the transformer,
```

The paper introduces the Vision Transformer (ViT), demonstrating that the standard Transformer architecture from natural language processing can be applied directly to images with minimal modifications.
* Patch Extraction and Embedding: Images are divided into fixed-size patches (e.g., $16\times 16$ pixels). Each patch is flattened into a 1D sequence and linearly projected to a constant latent vector size $D$. A learned 1D positional embedding is added to retain spatial information. Finally, a learnable [class] token is prepended to the sequence, and its final state serves as the global image representation for classification.
* Inductive Bias vs. Data Scale: Unlike Convolutional Neural Networks (CNNs), ViT lacks inherent 2D translation equivariance and local neighborhood structure. Consequently, it overfits and underperforms when trained on smaller datasets like ImageNet. However, when pre-trained on massive datasets (e.g., the 300 million images in JFT-300M), data scale trumps inductive bias, allowing ViT to match or exceed state-of-the-art CNNs.
* Attention Distance: ViT utilizes global multi-head self-attention, allowing it to integrate information across the entire image even in the lowest layers. The "attention distance" (conceptually analogous to a CNN's receptive field) scales with network depth, though some attention heads attend globally right from the start.
* Computational Efficiency: At scale, ViT delivers a superior performance-to-compute trade-off compared to large ResNets (like BiT), requiring substantially fewer TPUv3 core-days to pre-train for comparable accuracy.

In [ ]:
# model = torch.load('final_transformer.pth',weights_only=False)

In [ ]:
import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="application.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

In [ ]:
# torch.load('final_transformer.pth',weights_only=True)

In [ ]:
from dataclasses import dataclass

import torch
from torch import nn

import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="application.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


# def print(*args):
#     # Convert all inputs to strings and filter out empty values
#     string_elements = [str(arg) for arg in args]
#
#     # If more than 1 item was passed, concatenate them with a space
#     if len(string_elements) > 1:
#         log_message = " ".join(string_elements)
#     elif len(string_elements) == 1:
#         log_message = string_elements[0]
#     else:
#         return  # Skip empty print() calls entirely
#
#     # Send the final concatenated message straight to your colored logger
#     logger.info(log_message)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)


class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


# def averager(func):
#     num = 0
#     total = 0
#
#     def update(val, sz):
#         nonlocal num
#         nonlocal total
#         num += val * sz
#         total += sz
#
#     @wraps(func)
#     def inner():
#         nonlocal num, total
#         update(num / total)
#
#     return inner


def print_variance(name, data):
    neuron_variance = torch.mean(torch.var(data, dim=0))
    print(f"name={name!r}, Variance={neuron_variance.float()}")


class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        self.qkv = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        QKV = QKV.transpose(-2, -1)
        Q, K, V = QKV.chunk(3, dim=-1)
        A = Q @ K.transpose(-2, -1) / self.n_hidden**0.5
        scores = A
        if attn_mask is not None:
            "\n            causal masking future inputs of 0 to negative so that softmax will return 0\n"
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        A = torch.softmax(scores, dim=-1)
        Z = A @ V
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)
        Z = self.WO(Z)
        return (Z, A)


import torch
from torch import nn


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape
        qkv = self.qkv_projection(x)
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.n_hidden**0.5
        if attn_mask is not None:
            scores = scores.masked_fill(
                attn_mask.unsqueeze(1) == 0, float("-inf")
            )  # causal mask
        attn_alphas = torch.softmax(scores, dim=-1)
        context = attn_alphas @ v
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)
        Z = self.W0(A)
        return (Z, attn_alphas)


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        # self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )
        # self.norm2 = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, attn_mask=False
    ) -> tuple[torch.Tensor, torch.Tensor]:
        print_variance("Input", x)
        Z, A = self.attn(x=self.norm1(x), attn_mask=attn_mask)
        print_variance("after first attention block output", Z)
        x = Z + x
        print_variance("after residual 1 adding output to Z", x)
        ffn_out = self.ffn(self.norm2(x))
        x = ffn_out + x
        print_variance("after Applying FeedForward output with residual", x)
        print("Residual Block\n\n")
        return (x, A)


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self, x: torch.Tensor, attn_mask=False, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []
        Z = None
        A = []
        print(f"\n\n-------------------------Transformer: {x.shape}")
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)
            if return_attn is not None:
                A.append(alphas)
        if return_attn:
            return (x, torch.stack(A, dim=1))
        else:
            return (x, None)


class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.flattened_patch = (img_size // patch_size) ** 2
        self.patch_embedding = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.nout = nout

    def forward(self, x: torch.Tensor):
        """TODO: Implement the patch embedding. You want to split up the image into square patches of the given patch size. Then each patch_size x patch_size square should be linearly projected into an embedding of size nout. Hint: Take a look at nn.Conv2d. How can this be used to perform the patch embedding?"""
        out = self.patch_embedding(x)
        out = out.flatten(2)
        out = out.transpose(1, 2).long()
        logger.info(f"Output shape in patch embedding: {out.shape=}")
        return out


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        embs = self.patch_embed(img)
        B, T, _ = embs.shape
        logger.info(f"embs.shape={embs.shape!r}")
        pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        embs += self.pos_E(pos_ids).long()
        print_variance("Embeddings", embs.float())
        cls_token = self.cls_token.expand(len(embs), -1, -1)
        x = torch.cat([cls_token, embs], dim=1)
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        # print_variance("Output", x)
        out = self.head(x)[:, 0]
        print_variance("Projected Output", x)
        return (out, alphas)


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset, batch_size=256, shuffle=True, num_workers=2
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset, batch_size=256, shuffle=False, num_workers=2
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(
                model.state_dict(), "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
            )
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            print("Finished Training")
            break
        else:
            torch.save(
                model.state_dict(), "Vision_transformer_" + str(epoch + 1) + ".pt"
            )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
        print("Finished Training")

In [ ]:
class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.flattened_patch = (img_size // patch_size) ** 2
        self.patch_embedding = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.nout = nout

    def forward(self, x: torch.Tensor):
        """TODO: Implement the patch embedding. You want to split up the image into square patches of the given patch size. Then each patch_size x patch_size square should be linearly projected into an embedding of size nout. Hint: Take a look at nn.Conv2d. How can this be used to perform the patch embedding?"""
        out = self.patch_embedding(x)
        out = out.flatten(2)
        out = out.transpose(1, 2).long()
        logger.info(f"Output shape in patch embedding: {out.shape=}")
        return out

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        embs = self.patch_embed(img)
        B, T, _ = embs.shape
        logger.info(f"embs.shape={embs.shape!r}")
        pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        embs += self.pos_E(pos_ids).long()
        print_variance("Embeddings", embs.float())
        cls_token = self.cls_token.expand(len(embs), -1, -1)
        x = torch.cat([cls_token, embs], dim=1)
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        # print_variance("Output", x)
        out = self.head(x)[:, 0]
        print_variance("Projected Output", x)
        return (out, alphas)

In [ ]:
torch.arange(10)

In [ ]:
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = (AverageMeter(), AverageMeter())
        for img, labels in val_loader:
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return (loss_meter.calculate(), acc_meter.calculate())

In [ ]:
patch = PatchEmbed(32, 4, 3, 128)

In [ ]:
model = VisionTransformer(
    n_channels=3,
    nout=10,
    img_size=32,
    patch_size=4,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)

## Part 3.B Vision Tranformer

In [ ]:
print("Hi")

In [ ]:
class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


def print_variance(name, data):
    # First dimension(rows) is batch elements
    # Second dimension(columns) is neurons.
    # np_data = data.detach().numpy()

    # Compute variance across neurons and average these variances over members of the batch
    neuron_variance = torch.mean(torch.var(data, dim=0))
    # Print out the name and the variance
    print(f"{name=}, Variance={neuron_variance.float()}")


class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        # dim: the dimension of the input
        # n_hidden: the hidden dimensions for the attention layer
        # num_heads: the number of attention heads
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        # TODO: set up your parameters for multi-head attention. You should initialize
        #       num_heads attention heads (see nn.ModuleList) as well as a linear layer
        #       that projects the concatenated outputs of each head into dim
        #       (what size should this linear layer be?)

        # self.Wq = nn.Linear(dim,n_hidden)
        # self.Wk = nn.Linear(dim,n_hidden)
        # self.Wv = nn.Linear(dim,n_hidden)

        self.qkv = nn.Linear(
            dim, num_heads * n_hidden * 3, bias=False
        )  # Why bias false?

        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        # print(f"{QKV.shape=}")
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        # print(f"After reshaping reshaped: {QKV.shape=}")
        QKV = QKV.transpose(1, -2)
        Q, K, V = QKV.chunk(3, dim=-1)

        # First rescaling then applying attention mask if given
        A = Q @ K.transpose(-2, -1) / (self.n_hidden**0.5)

        scores = A

        if attn_mask is not None:
            """
            causal masking future inputs of 0 to negative so that softmax will return 0
            """
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        # Normalizing inputs across each layer
        A = torch.softmax(scores, dim=-1)

        # Calculating output for 1 head

        Z = A @ V

        # Concatenating outputs transpose and reshape along heads
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)

        # Final Projection
        Z = self.WO(Z)

        # return output, attention
        return Z, A


import torch.nn as nn


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden

        # 1. One combined projection layer for all heads at once
        # This replaces your nn.ModuleList loop completely
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)

        # 2. Final output projection (W0)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape  # Batch Size, Token Size, model_dim?

        # 3. Project all inputs for all heads in parallel
        # Shape becomes: (B, T, num_heads * n_hidden * 3)
        qkv = self.qkv_projection(x)

        # 4. Reshape and split into separate Query, Key, and Value tensors
        # New shape format: (B, num_heads, T, n_hidden)
        # 1. Split the final dimension into [3, num_heads, n_hidden]
        # qkv = qkv.reshape(B, T, 3, self.num_heads, self.n_hidden)
        # # 2. Permute to bring the 3-split and num_heads out front: (3, B, num_heads, T, n_hidden)
        # qkv = qkv.permute(2, 0, 1, 3, 4)
        # q, k, v = qkv[0], qkv[1], qkv[2]
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)

        # 5. Compute parallel attention weights (Alphas)
        # Shape: (B, num_heads, T, T)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.n_hidden**0.5)

        if attn_mask is not None:
            # Mask format: Convert 0s to a very large negative value (-1e9)
            # Unsqueeze mask to match (B, 1, T, T) for broadcasting across heads
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))

        attn_alphas = torch.softmax(scores, dim=-1)

        # 6. Apply weights to values
        # Shape: (B, num_heads, T, n_hidden)
        context = attn_alphas @ v

        # 7. Concatenate all heads back together efficiently
        # Permute back to (B, T, num_heads, n_hidden) -> Flatten to (B, T, num_heads * n_hidden)
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        # 8. Run through final output projection W0
        Z = self.W0(A)

        return Z, attn_alphas


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        super().__init__()

        # self.attn = MultiAttentionHead(dim, attn_dim, num_heads)
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        # self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )
        # self.norm2 = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, attn_mask=False
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
        #                  contains 1 if (in batch b) token i should attend on token j and 0
        #                  otherwise. shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      the attention weights of each of the attention heads.
        #                  shape: (B x Num_heads x T x T)
        print("\n\n--------- Attention Residual")
        print_variance("Input", x)
        Z, A = self.attn(x, attn_mask=attn_mask)
        # Z,A = attn_out,alphas
        print_variance("after first attention block output", Z)

        print("\n\n  In Attention Residual")
        print(f"{Z.shape=}\t,{x.shape=}\t.{A.shape=}\n")
        x = Z + x

        print_variance("after residual 1 adding output to Z", x)

        # ffn_out = F.gelu(self.ffn(self.norm2(x)))
        ffn_out = self.ffn(x)
        x = ffn_out + x

        print_variance("after Applying FeedForward output with residual", x)
        print_variance("Attention Variance:", A)
        print("Residual Block\n\n")
        print(f"{x.shape=}\n\n--------------")
        return x, A


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        # num_layers the number of attention layers.
        super().__init__()

        # ======= Answer START ========
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

        # ======= Answer END ========

    def forward(
        self, x: torch.Tensor, attn_mask=False, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []
        Z = None
        A = []
        # TODO: Implement the transformer forward pass! Pass the input successively through each of the AttentionResidual layers. If return_attn is True, collect the alphas along the way.

        # ======= Answer START ========
        print(f"\n\n-------------------------Transformer: {x.shape}")
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)

            # if return_attn is None:
            #     alphas = None
            # print(type(alphas), alphas.shape)
            if return_attn is not None:
                A.append(alphas)

        print(x.shape)
        print("return attention:", return_attn)
        # ======= Answer END ========
        if return_attn:
            return x, torch.stack(A, dim=1)
        else:
            return x, None

        # return x,None if   return_attn is False  else x, torch.stack(A,dim=1)
        # return output, collected_attns


class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        # img_size       the width and height of the image. you can assume that
        #                the images will be square
        # patch_size     the width of each square patch. You can assume that
        #                img_size is divisible by patch_size
        # nin            the number of input channels
        # nout           the number of output channels

        super().__init__()
        assert img_size % patch_size == 0

        self.img_size = img_size
        self.flattened_patch = (img_size // patch_size) ** 2

        # TODO Set up parameters for the Patch Embedding
        # ======= Answer START ========
        self.patch_embedding = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.nout = nout
        # self.projection = nn.Linear()

        # ======= Answer END ========

    def forward(self, x: torch.Tensor):
        # x        the input image. shape: (B, nin, Height, Width)
        #
        # Output
        # out      the patch embeddings for the input. shape: (B, num_patches, nout)

        """TODO: Implement the patch embedding. You want to split up the image into square patches of the given patch size. Then each patch_size x patch_size square should be linearly projected into an embedding of size nout. Hint: Take a look at nn.Conv2d. How can this be used to perform the patch embedding?"""
        # B, nin, H, W = x.shape
        # images = [[[[x * n + y + 1] for y in range(H)] for x in range(W)]]
        # patches = x.reshape(
        #     self.N*B*self.nout
        #
        # )

        out = self.patch_embedding(x)
        out = out.flatten(2)

        # ======= Answer START ========
        out = out.transpose(1, 2).long()
        print(f"Output shape in patch embedding: {out.shape=}")
        # ======= Answer END ========

        return out


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        # n_channels       number of input image channels
        # nout             desired output dimension
        # img_size         width of the square image
        # patch_size       width of the square patch
        # dim              embedding dimension
        # attn_dim         the hidden dimension of the attention layer
        # mlp_dim          the hidden layer dimension of the FFN
        # num_heads        the number of heads in the attention layer
        # num_layers       the number of attention layers.
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        self.pos_E = nn.Embedding(
            (img_size // patch_size) ** 2, dim
        )  # positional embedding matrix

        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))  # learned class embedding
        self.local_position_embedding = copy.deepcopy(self.pos_E)
        self.global_position_embedding = copy.deepcopy(self.cls_token)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.patch_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Projection Head

        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # img          the input image. shape: (B, nin, img_size, img_size)
        # return_attn  whether to return the attention alphas
        #
        # Outputs
        # out          the output of the vision transformer. shape: (B, nout)
        # alphas       the attention weights for all heads and layers. None if return_attn is False, otherwise
        #              shape: (B, num_layers, num_heads, num_patches + 1, num_patches + 1)

        # generate embeddings
        embs = self.patch_embed(img)  # patch embedding
        B, T, _ = embs.shape
        print(f"{embs.shape=}")
        # Positional Encoding
        pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        embs += self.pos_E(pos_ids).long()  # positional embedding
        # todo positional encoding such that there is no limit to sequence size..
        print("VIT\n\n\n")
        print_variance("Embeddings", embs.float())
        cls_token = self.cls_token.expand(len(embs), -1, -1)
        x = torch.cat([cls_token, embs], dim=1)

        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        # print_variance("Output Transformer:", x)
        out = self.head(x)[:, 0]
        print_variance("Projected Output after passing through projection head:", x)
        return out, alphas

In [ ]:
# Classify using the [CLS] token at index 0
#         out = self.head(x[:, 0])
#
#         return out, alphas

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 1,  # Number of global/CLS tokens
    ):
        # should be a mapping... and rotatory

        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)
        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas

# Gemini Spark Code

## Clearing Cachem

In [ ]:
def clean_memory_cache():
    """Safely flushes CPU and GPU memory cache pools across all platforms."""
    # 1. System.gc() equivalent: Cleans host/unified reference counts
    gc.collect()

    # 2. Safely check and flush NVIDIA CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 3. Safely check for MPS cache support without crashing
    elif torch.backends.mps.is_available():
        if hasattr(torch.mps, "empty_cache"):
            try:
                torch.mps.empty_cache()
            except Exception:
                # Silently bypass if the MPS backend refuses the call
                pass

In [5]:
device = torch.accelerator.current_accelerator().type

In [6]:
device

'mps'

In [16]:
from typing import Optional, Tuple, Any
import torch
import torch.nn as nn
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy

torch.autograd.set_detect_anomaly(True)

import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.w_o = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape

        # Shape: (B, T, 3 * num_heads * n_hidden) -> (B, num_heads, T, 3 * n_hidden)
        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        q, k, v = qkv.chunk(3, dim=-1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)
            scores = scores.masked_fill(attn_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        output = self.w_o(context)
        return output, attn_weights


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)

        # LayerNorm applied inside the FFN sequence only
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(x, attn_mask=attn_mask)
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(x)
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        collected_attns = []

        for layer in self.layers:
            x, alphas = layer(x, attn_mask=attn_mask)
            if return_attn:
                collected_attns.append(alphas)

        if return_attn:
            return x, torch.stack(collected_attns, dim=1)
        return x, None


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert (
            img_size % patch_size == 0
        ), "Image dimensions must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Output: (B, num_patches, nout) in float
        return self.proj(x).flatten(2).transpose(1, 2)


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, dim))

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification Head with its own LayerNorm
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # Prepend [CLS] token: (B, num_patches + 1, dim)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, embs), dim=1)

        # Add position embeddings
        x = x + self.pos_embed

        # Pass through Transformer encoder layers
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # Classify using the [CLS] token at index 0
        out = self.head(x[:, 0])

        return out, alphas


# class VisionTransformer(nn.Module):
#     def __init__(
#             self,
#             n_channels: int,
#             nout: int,
#             img_size: int,
#             patch_size: int,
#             dim: int,
#             attn_dim: int,
#             mlp_dim: int,
#             num_heads: int,
#             num_layers: int,
#             num_global_tokens: int = 1,  # Number of global/CLS tokens
#     ):
#         super().__init__()
#         self.num_global_tokens = num_global_tokens
#
#         self.patch_embed = PatchEmbed(
#             img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
#         )
#         num_patches = self.patch_embed.num_patches
#
#         # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)
#         self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))
#
#         # Position embeddings covering both global tokens and image patches
#         total_seq_len = num_global_tokens + num_patches
#         self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))
#
#         nn.init.trunc_normal_(self.cls_tokens, std=0.02)
#         nn.init.trunc_normal_(self.pos_embed, std=0.02)
#
#         self.transformer = Transformer(
#             dim=dim,
#             attn_dim=attn_dim,
#             mlp_dim=mlp_dim,
#             num_heads=num_heads,
#             num_layers=num_layers,
#         )
#
#         # Classification / Projection Head
#         self.head = nn.Sequential(
#             nn.LayerNorm(dim),
#             nn.Linear(dim, nout),
#         )
#
#     def forward(
#             self, img: torch.Tensor, return_attn: bool = False
#     ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
#         B = img.shape[0]
#
#         # 1. Patch embeddings: (B, num_patches, dim)
#         embs = self.patch_embed(img)
#
#         # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
#         cls_tokens = self.cls_tokens.expand(B, -1, -1)
#
#         # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
#         x = torch.cat((cls_tokens, embs), dim=1)
#
#         # 4. Add position embeddings
#         x = x + self.pos_embed
#
#         # 5. Transformer forward pass
#         x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
#
#         # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
#         global_repr = x[:, : self.num_global_tokens]
#
#         # Aggregate global tokens:
#         # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
#         # out = self.head(global_repr[:, 0])
#         #
#         # - Option B: Mean-pool across all global tokens
#         out = self.head(global_repr.mean(dim=1))  # (B, nout)
#
#         return out, alphas
# class VisionTransformer2(nn.Module):
#     def __init__(
#         self,
#         n_channels: int,
#         nout: int,
#         img_size: int,
#         patch_size: int,
#         dim: int,
#         attn_dim: int,
#         mlp_dim: int,
#         num_heads: int,
#         num_layers: int,
#     ):
#         super().__init__()
#         self.patch_embed = PatchEmbed(
#             img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
#         )
#         self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)
#         self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
#         self.transformer = Transformer(
#             dim=dim,
#             attn_dim=attn_dim,
#             mlp_dim=mlp_dim,
#             num_heads=num_heads,
#             num_layers=num_layers,
#         )
#         self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))
#
#     def forward(
#         self, img: torch.Tensor, return_attn=False
#     ) -> tuple[torch.Tensor, torch.Tensor | None]:
#         embs = self.patch_embed(img)
#         B, T, _ = embs.shape
#         logger.info(f"embs.shape={embs.shape!r}")
#
#         pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
#         embs = embs + self.pos_E(pos_ids).long()
#         print_variance("Embeddings", embs.float())
#         cls_token = self.cls_token.expand(len(embs), -1, -1)
#         x = torch.cat([cls_token, embs], dim=1)
#         x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
#         # print_variance("Output", x)
#         out = self.head(x)[:, 0]
#         print_variance("Projected Output", x)
#         return (out, alphas)

## Global Tokens in Vision Transformer for infinite sequence length

In [17]:
# Time Estimate: less than 5 minutes on T4 GPU
# train the model
import tqdm

result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])

In [20]:
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()

In [187]:
from typing import Optional, Tuple, Any
import torch
import torch.nn as nn
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy

torch.autograd.set_detect_anomaly(True)

import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.w_o = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape

        # Shape: (B, T, 3 * num_heads * n_hidden) -> (B, num_heads, T, 3 * n_hidden)
        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        q, k, v = qkv.chunk(3, dim=-1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)
            scores = scores.masked_fill(attn_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        output = self.w_o(context)
        return output, attn_weights


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)

        # LayerNorm applied inside the FFN sequence only
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(x, attn_mask=attn_mask)
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(x)
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        collected_attns = []

        for layer in self.layers:
            x, alphas = layer(x, attn_mask=attn_mask)
            if return_attn:
                collected_attns.append(alphas)

        if return_attn:
            return x, torch.stack(collected_attns, dim=1)
        return x, None


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert (
            img_size % patch_size == 0
        ), "Image dimensions must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Output: (B, num_patches, nout) in float
        return self.proj(x).flatten(2).transpose(1, 2)


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 1,  # Number of global/CLS tokens
    ):
        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)

        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas

## Cache Clearing,

In [22]:
import gc

gc.collect()
import gc
import torch


def clean_memory_cache():
    """Safely flushes CPU and GPU memory cache pools across all platforms."""
    # 1. System.gc() equivalent: Cleans host/unified reference counts
    gc.collect()

    # 2. Safely check and flush NVIDIA CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 3. Safely check for MPS cache support without crashing
    elif torch.backends.mps.is_available():
        if hasattr(torch.mps, "empty_cache"):
            try:
                torch.mps.empty_cache()
            except Exception:
                # Silently bypass if the MPS backend refuses the call
                pass

In [23]:
import tqdm


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)

    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset, batch_size=256, shuffle=True, num_workers=2
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset, batch_size=256, shuffle=False, num_workers=2
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
    )
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()

        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )

        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)

        # if early_stopping.step(val_loss,model,epoch=epoch):
        #     early_stopping.restore_best_weights(model)

        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(
                model.state_dict(), "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
            )
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            # print("Finished Training")

        else:
            torch.save(
                model.state_dict(), "Vision_transformer_" + str(epoch + 1) + ".pt"
            )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
        clean_memory_cache()
    print("Finished Training")

## Test 2

In [24]:
main()  # time taken 14 minutes 17 seconds

Training at 0: 100%|██████████| 196/196 [00:56<00:00,  3.46it/s]


Train Epoch: 0, Loss: 1.727633017539978, Acc: 0.3611


W0912 18:36:54.730000 12690 torch/_inductor/utils.py:1953] [1/0] Not enough SMs to use max_autotune_gemm mode


Val Epoch: 0, Loss: 1.5280103694915772, Acc: 0.4363
Val Epoch: 1, Loss: 1.5280103694915772, Acc: 0.4363


Training at 1: 100%|██████████| 196/196 [00:53<00:00,  3.66it/s]

...

Train Epoch: 8, Loss: 0.47866997610092166, Acc: 0.8379600000190734


Val Epoch: 8, Loss: 0.8273304260253906, Acc: 0.7244
Val Epoch: 9, Loss: 0.8273304260253906, Acc: 0.7244
Val Epoch: 9, Loss: 0.8273304260253906, Acc: 0.7244


Training at 9: 100%|██████████| 196/196 [00:53<00:00,  3.64it/s]

Train Epoch: 9, Loss: 0.4276150000858307, Acc: 0.8563399999809265


Val Epoch: 9, Loss: 0.818748565864563, Acc: 0.7281
Val Epoch: 10, Loss: 0.818748565864563, Acc: 0.7281
Finished Training


In [ ]:
main()  # time taken 9 minutes 54 seconds only print statements

In [28]:
import pandas as pd

df = pd.DataFrame.from_dict(
    {
        "train_acc": result.train_accs,
        "train_loss": result.train_losses,
        "val_acc": result.val_accs,
        "val_loss": result.val_losses,
    }
)

In [29]:
df

,train_acc,train_loss,val_acc,val_loss


In [ ]:
import pytorch_lightning
from pytorch_lightning import utilities

dir(utilities)

## Without Normalization scaled tanx

In [ ]:
class DyT(nn.Module):
    def __init__(self, num_features, alpha_init_value=0.5):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1) * alpha_init_value)
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))

    def forward(self, x):
        x = torch.tanh(self.alpha * x)
        return x * self.weight + self.bias

In [ ]:
import pytorch_lightning as pl
from torchmetrics import Accuracy,Recall,Specificity,R2Score,Precision,
import torchmetrics
dir(torchmetrics)

In [ ]:
# class MLP(pl.LightningModule):
#     def __init__(self, *args: Any, **kwargs: Any):
#         super().__init__()
#         self.train_acc = Accuracy()
#         self.sensitivitu =

In [ ]:
import torchmetrics

dir(torchmetrics)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/

In [ ]:
# class VisionTransformer(nn.Module):
#     def __init__(
#             self,
#             n_channels: int,
#             nout: int,
#             img_size: int,
#             patch_size: int,
#             dim: int,
#             attn_dim: int,
#             mlp_dim: int,
#             num_heads: int,
#             num_layers: int,
#     ):
#         # n_channels       number of input image channels
#         # nout             desired output dimension
#         # img_size         width of the square image
#         # patch_size       width of the square patch
#         # dim              embedding dimension
#         # attn_dim         the hidden dimension of the attention layer
#         # mlp_dim          the hidden layer dimension of the FFN
#         # num_heads        the number of heads in the attention layer
#         # num_layers       the number of attention layers.
#         super().__init__()
#         self.patch_embed = PatchEmbed(
#             img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
#         )
#         self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)  # positional embedding matrix
#
#         self.cls_token = nn.Parameter(torch.randn(1, 1, dim))  # learned class embedding
#         self.transformer = Transformer(
#             dim=dim,
#             attn_dim=attn_dim,
#             mlp_dim=mlp_dim,
#             num_heads=num_heads,
#             num_layers=num_layers,
#         )
#
#         # Projection Head
#
#         self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))
#
#     def forward(
#             self, img: torch.Tensor, return_attn=False
#     ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
#         # img          the input image. shape: (B, nin, img_size, img_size)
#         # return_attn  whether to return the attention alphas
#         #
#         # Outputs
#         # out          the output of the vision transformer. shape: (B, nout)
#         # alphas       the attention weights for all heads and layers. None if return_attn is False, otherwise
#         #              shape: (B, num_layers, num_heads, num_patches + 1, num_patches + 1)
#
#         # generate embeddings
#         embs = self.patch_embed(img)  # patch embedding
#         B, T, _ = embs.shape
#         print(f"{embs.shape=}")
#         # Positional Encoding
#         pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
#         embs += self.pos_E(pos_ids).long()  # positional embedding
#         #todo positional encoding such that there is no limit to sequence size..
#         print("VIT\n\n\n")
#         print_variance("Embeddings", embs.float())
#         cls_token = self.cls_token.expand(len(embs), -1, -1)
#         x = torch.cat([cls_token, embs], dim=1)
#
#         x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
#         #print_variance("Output Transformer:", x)
#         out = self.head(x)[:, 0]
#         print_variance("Projected Output after passing through projection head:", x)
#         return out, alphas

# Vision Transformer Deven

In [ ]:
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy

import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


def print(*args):
    # Convert all inputs to strings and filter out empty values
    string_elements = [str(arg) for arg in args]

    # If more than 1 item was passed, concatenate them with a space
    if len(string_elements) > 1:
        log_message = " ".join(string_elements)
    elif len(string_elements) == 1:
        log_message = string_elements[0]
    else:
        return  # Skip empty print() calls entirely

    # Send the final concatenated message straight to your colored logger
    logger.info(log_message)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)


class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


# def averager(func):
#     num = 0
#     total = 0
#
#     def update(val, sz):
#         nonlocal num
#         nonlocal total
#         num += val * sz
#         total += sz
#
#     @wraps(func)
#     def inner():
#         nonlocal num, total
#         update(num / total)
#
#     return inner


def print_variance(name, data):
    neuron_variance = torch.mean(torch.var(data, dim=0))
    print(f"name={name!r}, Variance={neuron_variance.float()}")


class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        self.qkv = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        QKV = QKV.transpose(-2, -1)
        Q, K, V = QKV.chunk(3, dim=-1)
        A = Q @ K.transpose(-2, -1) / self.n_hidden**0.5
        scores = A
        if attn_mask is not None:
            "\n            causal masking future inputs of 0 to negative so that softmax will return 0\n"
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        A = torch.softmax(scores, dim=-1)
        Z = A @ V
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)
        Z = self.WO(Z)
        return (Z, A)


import torch
from torch import nn


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape
        qkv = self.qkv_projection(x)
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.n_hidden**0.5
        if attn_mask is not None:
            scores = scores.masked_fill(
                attn_mask.unsqueeze(1) == 0, float("-inf")
            )  # causal mask
        attn_alphas = torch.softmax(scores, dim=-1)
        context = attn_alphas @ v
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)
        Z = self.W0(A)
        return (Z, attn_alphas)


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        # self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )
        # self.norm2 = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, attn_mask=False
    ) -> tuple[torch.Tensor, torch.Tensor]:
        print_variance("Input", x)
        Z, A = self.attn(x, attn_mask=attn_mask)
        print_variance("after first attention block output", Z)
        x = Z + x
        print_variance("after residual 1 adding output to Z", x)
        ffn_out = self.ffn(x)
        x = ffn_out + x
        print_variance("after Applying FeedForward output with residual", x)
        print("Residual Block\n\n")
        return (x, A)


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self, x: torch.Tensor, attn_mask=False, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []
        Z = None
        A = []
        print(f"\n\n-------------------------Transformer: {x.shape}")
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)
            if return_attn is not None:
                A.append(alphas)
        if return_attn:
            return (x, torch.stack(A, dim=1))
        else:
            return (x, None)


class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.flattened_patch = (img_size // patch_size) ** 2
        self.patch_embedding = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.nout = nout

    def forward(self, x: torch.Tensor):
        """TODO: Implement the patch embedding. You want to split up the image into square patches of the given patch size. Then each patch_size x patch_size square should be linearly projected into an embedding of size nout. Hint: Take a look at nn.Conv2d. How can this be used to perform the patch embedding?"""
        out = self.patch_embedding(x)
        out = out.flatten(2)
        out = out.transpose(1, 2).long()
        logger.info(f"Output shape in patch embedding: {out.shape=}")
        return out


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        total_patches = self.patch_embed.flattened_patch

        # self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.positional_embeddings = nn.Parameter(
            torch.zeros(1, total_patches + 1, dim)
        )
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.positional_embeddings, std=0.02)
        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        embs = self.patch_embed(img)
        B, T, _ = embs.shape
        # logger.info(f"embs.shape={embs.shape!r}")
        # Getting embeddings from tokens
        embs = self.patch_embed(img)

        # Adding a cls token
        cls_token = self.cls_token.expand(B, -1, -1)

        x = torch.cat([embs, cls_token], dim=1)

        x = x + self.positional_embeddings
        # pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        # embs += self.pos_E(pos_ids).long()
        # print_variance("Embeddings", embs.float())

        # x = torch.cat([cls_token, embs], dim=1)
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        # print_variance("Output", x)
        out = self.head(x)[:, 0]
        # print_variance("Projected Output", x)
        return (out, alphas)


result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
import os

checkpoints = os.makedirs("vit", exist_ok=True)
checkpoint_dir = "vit"

import tqdm


@dataclass(init=True)
class EarlyStopping:

    patience: int = 3
    min_delta: float = 0.001
    mode: str = "min"
    counter: int = 0
    best_score: float = float("inf")
    best_weights = None
    best_epoch = 0

    def step(self, val_metric: float, model: nn.Module, epoch: int) -> bool:
        if self.mode == "max":
            self.best_score = float("-inf")
            improved_condition = val_metric > (self.best_score + self.min_delta)
        else:
            improved_condition = val_metric < (self.best_score - self.min_delta)

        if improved_condition:
            best_save_path = os.path.join(
                checkpoint_dir, "Best_Model_" + str(epoch) + ".pt"
            )
            self.best_score = val_metric
            self.best_weights = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), best_save_path)
            self.best_epoch = epoch
            self.counter = 0  # Reset

        else:
            self.counter += 1

        return True if self.patience >= self.counter else False

    def restore_best_weights(self, model: nn.Module) -> None:
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)
            print(
                f"Restored best weigths at epoch {self.best_epoch}, Best Metric for condition: {self.mode.capitalize()}: {self.best_score:.6f}"
            )


@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()


early_stopping = EarlyStopping(patience=3, min_delta=0.03, mode="min")

import tqdm


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)

    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset, batch_size=256, shuffle=True, num_workers=2
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset, batch_size=256, shuffle=False, num_workers=2
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
    )
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()

        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )

        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)

        if early_stopping.step(val_loss, model, epoch=epoch):
            early_stopping.restore_best_weights(model)

        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(
                model.state_dict(), "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
            )
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            # print("Finished Training")

        else:
            torch.save(
                model.state_dict(), "Vision_transformer_" + str(epoch + 1) + ".pt"
            )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
    print("Finished Training")

In [27]:
checkpoints = os.makedirs("vit", exist_ok=True)
checkpoint_dir = "vit"

In [ ]:
main()  # time taken 13 minutes 14 seconds why?

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(
    {
        "train_acc": result.train_accs,
        "train_loss": result.train_losses,
        "val_acc": result.val_accs,
        "val_loss": result.val_losses,
    }
)

In [ ]:
df

## DEBUG

In [ ]:
x = torch.tensor(1)
torch.arange(10).to(x).long()

In [ ]:
import os

os.cpu_count()

In [ ]:
# set up the dataset and dataloader
import os

MEAN = [0.4914, 0.4822, 0.4465]
STD = [0.2470, 0.2435, 0.2616]
img_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]
)
inv_transform = transforms.Compose(
    [
        transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
        transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
        transforms.ToPILImage(),
    ]
)

train_dataset = torchvision.datasets.CIFAR10(
    train=True, root="data", transform=img_transform, download=True
)
val_dataset = torchvision.datasets.CIFAR10(
    train=False, root="data", transform=img_transform
)

num_workers = min(4, os.cpu_count())

train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    pin_memory=True,  # Keep active for maximum PCIe transfer speeds
    # Keeps worker processes alive to prevent rebuilding cache pools
)
val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    pin_memory=True,
)

In [ ]:
len(train_dataset), len(val_dataset)

In [ ]:
len(train_dataloader), len(val_dataloader)

In [ ]:
model = VisionTransformer(
    n_channels=3,
    nout=10,
    img_size=32,
    patch_size=4,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)

In [ ]:
img, label = next(iter(train_dataloader))

In [ ]:
img.shape

In [ ]:
out, alpha = model(img.to(device))

In [ ]:
out.shape

In [ ]:
print_variance("Output:", out)

## Part 3.C

In [ ]:
# set up the dataset and dataloader
from torchvision import transforms
import torchvision
import numpy as np

MEAN = [0.4914, 0.4822, 0.4465]
STD = [0.2470, 0.2435, 0.2616]
img_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]
)
inv_transform = transforms.Compose(
    [
        transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
        transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
        transforms.ToPILImage(),
    ]
)

train_dataset = torchvision.datasets.CIFAR10(
    train=True, root="data", transform=img_transform, download=True
)
val_dataset = torchvision.datasets.CIFAR10(
    train=False, root="data", transform=img_transform
)
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
)
val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
)

In [26]:
device = torch.accelerator.current_accelerator().type

In [ ]:
len(train_dataloader)

In [ ]:
img, label = next(iter(train_dataloader))

In [ ]:
img = img.to(device)

In [ ]:
img.shape

In [ ]:
# torchvision.io.read_image

In [ ]:
class_names = train_dataset.classes

In [ ]:
type(img)

In [ ]:
data, class_name = train_dataset[0]

In [ ]:
data[0, :, :]

In [ ]:
import matplotlib.pyplot as plt


def visualize_tensor_data(data: torch.Tensor, label: int):
    # Data is a tensor of shape [C, W, H]  (C is the channel dimension, 3 for RGB)
    # Put channel at last
    data = data.permute(1, 2, 0)
    print(f"{data.shape=}")
    # Un-normalize
    data = data * torch.as_tensor(STD) + torch.as_tensor(MEAN)
    plt.imshow(data)
    plt.axis("off")
    plt.title(f"Label = {class_names[label]}")
    plt.savefig(class_names[label])


visualize_tensor_data(data, class_name)

In [ ]:
class_name

## Model Training

In [ ]:
# set up the model and optimizer

import torch.optim as optim

model = VisionTransformer(
    n_channels=3,
    nout=10,
    img_size=32,
    patch_size=4,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)

criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 10
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
)

In [ ]:
model

In [ ]:
import torch


# todo model should generalize not overfit or underfit
class EarlyStopping:
    def __init__(
        self,
        patience: int = 3,
        min_delta: float = 0.001,
        mode: str = "min",
        max_epoch: int = 8,
    ):
        """
        Args:
            patience: Number of epochs with no improvement before stopping.
            min_delta: Minimum change to qualify as an improvement.
            mode: 'min' for loss, 'max' for accuracy.
            max_epoch: Hard upper limit on epoch index (stops at epoch 8).
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.max_epoch = max_epoch
        self.counter = 0
        self.best_score = float("inf") if mode == "min" else float("-inf")
        self.best_weights = None
        self.best_epoch = 0

    def step(self, val_metric: float, model: torch.nn.Module, epoch: int) -> bool:
        # Check for improvement
        if self.mode == "min":
            improved = val_metric < (self.best_score - self.min_delta)
        else:
            improved = val_metric > (self.best_score + self.min_delta)

        if improved:
            self.best_score = val_metric
            self.best_weights = copy.deepcopy(model.state_dict())
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1

        # Check stopping criteria: patience limit OR reached target epoch 8
        if epoch >= self.max_epoch or self.counter >= self.patience:
            return True
        return False

    def restore_best_weights(self, model: torch.nn.Module):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)
            print(
                f"Restored best weights from Epoch {self.best_epoch} "
                f"(Best Val {self.mode.capitalize()}: {self.best_score:.4f})"
            )

In [ ]:
model(img.to(device))

In [ ]:
# model.load_state_dict(torch.load("checkpoints/Vision_transformer_4.pt", weights_only=True))

In [ ]:
list(model.state_dict().keys())

In [ ]:
list(x.shape for x in model.state_dict().values())

In [ ]:
list(x.requires_grad for x in model.state_dict().values())

In [ ]:
# model.train()

In [ ]:
for parameter, w in model.named_parameters():
    print(f"{parameter=},with {w.shape}")

In [ ]:
# evaluate the model
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()

In [ ]:
from dataclasses import dataclass


@dataclass(init=True)
class TrainResult:
    r"""
    A collection containing everything we need to know about the training results
    """

    # num_epochs: int

    # Training loss (saved at each iteration in `train_epoch`)
    train_losses: List[float]

    # The epochs where we perform evaluation
    # eval_epochs: List[int]

    # Training accuracies, computed at each epoch in `eval_epochs`
    train_accs: List[float]

    # Validation accuracies, computed at each epoch in `eval_epochs`
    val_accs: List[float]
    val_losses: List[float]
    # The last validation evaluation full result
    # final_val_eval_result: EvaluateResult = None

In [ ]:
import copy

model1 = copy.deepcopy(model)

In [ ]:
import os

checkpoints = os.makedirs("checkpoints", exist_ok=True)
checkpoints = "checkpoints"
# filename = f"Vision_transformer_Best_{epoch + 1}.pt"

In [24]:
import gc
import torch


def clean_memory_cache():
    """Safely flushes CPU and GPU memory cache pools across all platforms."""
    # 1. System.gc() equivalent: Cleans host/unified reference counts
    gc.collect()

    # 2. Safely check and flush NVIDIA CUDA cache
    if torch.cuda.is_available():
        print(f"{torch.cuda.memory_allocated()/(1024*1024):.2f} MB")

        torch.cuda.empty_cache()

    # 3. Safely check for MPS cache support without crashing
    elif torch.backends.mps.is_available():
        mps_mem = torch.mps.driver_allocated_memory() / (1024 * 1024)
        print(f"MPS Alloc: {mps_mem:.2f} MB")
        if hasattr(torch.mps, "empty_cache"):
            try:
                torch.mps.empty_cache()
            except Exception:
                # Silently bypass if the MPS backend refuses the call
                pass

In [69]:
hasattr(torch.mps, "empty_cache")

True

In [25]:
clean_memory_cache()

21.20 MB


In [ ]:
torch.mps.empty_cache()

In [15]:
torch.mps.driver_allocated_memory() / 1024 / 1024

17.6875

In [ ]:
# Time Estimate: less than 5 minutes on T4 GPU
# train the model
import tqdm

result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])

for epoch in range(NUM_EPOCHS):  #
    loss_meter = AverageMeter()
    acc_meter = AverageMeter()

    for img, labels in tqdm.tqdm(train_dataloader, desc="Training at " + str(epoch)):
        img, labels = img.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs, _ = model(img)
        loss = criterion(outputs, labels)
        loss_meter.update(loss.item(), len(img))
        acc = (outputs.argmax(-1) == labels).float().mean().item()
        acc_meter.update(acc, len(img))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    scheduler.step()
    result.train_losses.append(loss_meter.calculate())
    result.train_accs.append(acc_meter.calculate())
    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
    )
    val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
    result.val_losses.append(val_loss)
    result.val_accs.append(val_acc)
    print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
    filename = f"Vision_transformer_Best_{epoch + 1}.pt"
    save_path = os.path.join(checkpoints, filename)
    if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
        torch.save(model.state_dict(), save_path)
    else:
        torch.save(model.state_dict(), "Vision_transformer_" + str(epoch + 1) + ".pt")
    clean_memory_cache()
val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
print("Finished Training")

In [ ]:
clean_memory_cache()

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(
    {
        "train_acc": result.train_accs,
        "train_loss": result.train_losses,
        "val_acc": result.val_accs,
        "val_loss": result.val_losses,
    }
)

In [ ]:
df

In [ ]:
df.to_csv("results.csv")

### Loading model

In [ ]:
model.load_state_dict(torch.load("Vision_transformer_10.pt", weights_only=True))

In [ ]:
for key, value in model.state_dict().items():
    print(f"{key=},{value.shape}")

# Part 3.D

In [ ]:
for val_batch in val_dataloader:
    break

model.eval()
with torch.no_grad():
    img, labels = val_batch
    img = img.to(device)
    outputs, attns = model(img, return_attn=True)

fig, ax = plt.subplots(2, 20, figsize=(20, 2))
for i in range(20):
    flattened_attns = (
        attns.flatten(1, 2)[:, :, 0, 1:].mean(1).reshape(-1, 8, 8).cpu().numpy()
    )
    # ax[0, i].imshow(inv_transform(img[i]))
    # img = inv_transform(img[i].cpu())

    ax[0, i].imshow(inv_transform(img[i]))
    # plt.savefig(inv_transform(img[i]),'img'+str(i)+'.png')

    ax[1, i].imshow(flattened_attns[i])

    ax[0, i].axis(False)
    ax[1, i].axis(False)

## Generative Adversarial Examples

In [ ]:
dir(img)

In [ ]:
from PIL import Image

img = inv_transform(img).cpu()
Image.open(img)

In [ ]:
class LowerLayerPerturbation:
    def __init__(self, model, target_layer, epsilon=0.03):
        self.model = model
        self.target_layer = target_layer
        self.epsilon = epsilon
        self.layer_output = None

        # Register a forward hook to grab the lower layer's features
        self.hook = self.target_layer.register_forward_hook(self.get_features)

    def get_features(self, module, input, output):
        # Save intermediate feature representations
        self.layer_output = output

    def generate_attack(self, x):
        # Force PyTorch to track gradients on the input image
        x = x.clone().detach().requires_grad_(True)

        # Pass input through the network to trigger the hook
        _ = self.model(x)

        # Define an objective on the lower layer features (e.g., maximize variance/norm)
        # This disrupts the foundational feature extraction process
        loss = torch.norm(self.layer_output)

        # Backpropagate to the input image
        loss.backward()

        # Create the adversarial perturbation using the sign of the gradient
        with torch.no_grad():
            perturbation = self.epsilon * x.grad.sign()
            adversarial_x = x + perturbation
            # Keep pixel values valid
            adversarial_x = torch.clamp(adversarial_x, 0.0, 1.0)

        return adversarial_x

    def remove_hook(self):
        # Clean up the hook to free memory
        self.hook.remove()


# --- Usage Example ---
# if __name__ == "__main__":
#     # Create a simple dummy model
#     model = nn.Sequential(
#         nn.Conv2d(3, 16, kernel_size=3, padding=1), # Target lower layer
#         nn.ReLU(),
#         nn.Flatten(),
#         nn.Linear(16 * 32 * 32, 10)
#     ).eval()
#
#     # Initialize attacker targeting the first layer
#     attacker = LowerLayerPerturbation(model, model[0], epsilon=0.02)
#
#     # Generate a random dummy image (Batch size 1, 3 channels, 32x32)
#     dummy_image = torch.rand(1, 3, 32, 32)
#
#     # Create the perturbed image
#     adv_image = attacker.generate_attack(dummy_image)
#     print("Original shape:", dummy_image.shape)
#     print("Adversarial shape:", adv_image.shape)
#
#     # Clean up
#     attacker.remove_hook()

In [ ]:
dummy = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(16 * 32 * 32, 10),
).eval()

In [ ]:
dummy[0]

In [ ]:
list(model.named_modules())

In [ ]:
model.get_submodule("transformer")

In [ ]:
class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


def print_variance(name, data):
    # First dimension(rows) is batch elements
    # Second dimension(columns) is neurons.
    # np_data = data.detach().numpy()

    # Compute variance across neurons and average these variances over members of the batch
    neuron_variance = torch.mean(torch.var(data, dim=0))
    # Print out the name and the variance
    print(f"{name=}, Variance={neuron_variance.float()}")


class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        # dim: the dimension of the input
        # n_hidden: the hidden dimensions for the attention layer
        # num_heads: the number of attention heads
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        # TODO: set up your parameters for multi-head attention. You should initialize
        #       num_heads attention heads (see nn.ModuleList) as well as a linear layer
        #       that projects the concatenated outputs of each head into dim
        #       (what size should this linear layer be?)

        # self.Wq = nn.Linear(dim,n_hidden)
        # self.Wk = nn.Linear(dim,n_hidden)
        # self.Wv = nn.Linear(dim,n_hidden)

        self.qkv = nn.Linear(
            dim, num_heads * n_hidden * 3, bias=False
        )  # Why bias false?

        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        # print(f"{QKV.shape=}")
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        # print(f"After reshaping reshaped: {QKV.shape=}")
        QKV = QKV.transpose(1, -2)
        Q, K, V = QKV.chunk(3, dim=-1)

        # First rescaling then applying attention mask if given
        A = Q @ K.transpose(-2, -1) / (self.n_hidden**0.5)

        scores = A

        if attn_mask is not None:
            """
            causal masking future inputs of 0 to negative so that softmax will return 0
            """
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        # Normalizing inputs across each layer
        A = torch.softmax(scores, dim=-1)

        # Calculating output for 1 head

        Z = A @ V

        # Concatenating outputs transpose and reshape along heads
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)

        # Final Projection
        Z = self.WO(Z)

        # return output, attention
        return Z, A


import torch.nn as nn


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden

        # 1. One combined projection layer for all heads at once
        # This replaces your nn.ModuleList loop completely
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)

        # 2. Final output projection (W0)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape  # Batch Size, Token Size, model_dim?

        # 3. Project all inputs for all heads in parallel
        # Shape becomes: (B, T, num_heads * n_hidden * 3)
        qkv = self.qkv_projection(x)

        # 4. Reshape and split into separate Query, Key, and Value tensors
        # New shape format: (B, num_heads, T, n_hidden)
        # 1. Split the final dimension into [3, num_heads, n_hidden]
        # qkv = qkv.reshape(B, T, 3, self.num_heads, self.n_hidden)
        # # 2. Permute to bring the 3-split and num_heads out front: (3, B, num_heads, T, n_hidden)
        # qkv = qkv.permute(2, 0, 1, 3, 4)
        # q, k, v = qkv[0], qkv[1], qkv[2]
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)

        # 5. Compute parallel attention weights (Alphas)
        # Shape: (B, num_heads, T, T)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.n_hidden**0.5)

        if attn_mask is not None:
            # Mask format: Convert 0s to a very large negative value (-1e9)
            # Unsqueeze mask to match (B, 1, T, T) for broadcasting across heads
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))

        attn_alphas = torch.softmax(scores, dim=-1)

        # 6. Apply weights to values
        # Shape: (B, num_heads, T, n_hidden)
        context = attn_alphas @ v

        # 7. Concatenate all heads back together efficiently
        # Permute back to (B, T, num_heads, n_hidden) -> Flatten to (B, T, num_heads * n_hidden)
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        # 8. Run through final output projection W0
        Z = self.W0(A)

        return Z, attn_alphas


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        super().__init__()

        # self.attn = MultiAttentionHead(dim, attn_dim, num_heads)
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        # self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )
        # self.norm2 = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, attn_mask=False
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
        #                  contains 1 if (in batch b) token i should attend on token j and 0
        #                  otherwise. shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      the attention weights of each of the attention heads.
        #                  shape: (B x Num_heads x T x T)
        print("\n\n--------- Attention Residual")
        print_variance("Input", x)
        Z, A = self.attn(x, attn_mask=attn_mask)
        # Z,A = attn_out,alphas
        print_variance("after first attention block output", Z)

        print("\n\n  In Attention Residual")
        print(f"{Z.shape=}\t,{x.shape=}\t.{A.shape=}\n")
        x = Z + x

        print_variance("after residual 1 adding output to Z", x)

        # ffn_out = F.gelu(self.ffn(self.norm2(x)))
        ffn_out = self.ffn(x)
        x = ffn_out + x

        print_variance("after Applying FeedForward output with residual", x)
        print_variance("Attention Variance:", A)
        print("Residual Block\n\n")
        print(f"{x.shape=}\n\n--------------")
        return x, A


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        # num_layers the number of attention layers.
        super().__init__()

        # ======= Answer START ========
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

        # ======= Answer END ========

    def forward(
        self, x: torch.Tensor, attn_mask=False, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []
        Z = None
        A = []
        # TODO: Implement the transformer forward pass! Pass the input successively through each of the AttentionResidual layers. If return_attn is True, collect the alphas along the way.

        # ======= Answer START ========
        print(f"\n\n-------------------------Transformer: {x.shape}")
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)

            # if return_attn is None:
            #     alphas = None
            # print(type(alphas), alphas.shape)
            if return_attn is not None:
                A.append(alphas)

        print(x.shape)
        print("return attention:", return_attn)
        # ======= Answer END ========
        if return_attn:
            return x, torch.stack(A, dim=1)
        else:
            return x, None

        # return x,None if   return_attn is False  else x, torch.stack(A,dim=1)
        # return output, collected_attns


class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        # img_size       the width and height of the image. you can assume that
        #                the images will be square
        # patch_size     the width of each square patch. You can assume that
        #                img_size is divisible by patch_size
        # nin            the number of input channels
        # nout           the number of output channels

        super().__init__()
        assert img_size % patch_size == 0

        self.img_size = img_size
        self.flattened_patch = (img_size // patch_size) ** 2

        # TODO Set up parameters for the Patch Embedding
        # ======= Answer START ========
        self.patch_embedding = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.nout = nout
        # self.projection = nn.Linear()

        # ======= Answer END ========

    def forward(self, x: torch.Tensor):
        # x        the input image. shape: (B, nin, Height, Width)
        #
        # Output
        # out      the patch embeddings for the input. shape: (B, num_patches, nout)

        """TODO: Implement the patch embedding. You want to split up the image into square patches of the given patch size. Then each patch_size x patch_size square should be linearly projected into an embedding of size nout. Hint: Take a look at nn.Conv2d. How can this be used to perform the patch embedding?"""
        # B, nin, H, W = x.shape
        # images = [[[[x * n + y + 1] for y in range(H)] for x in range(W)]]
        # patches = x.reshape(
        #     self.N*B*self.nout
        #
        # )

        out = self.patch_embedding(x)
        out = out.flatten(2)

        # ======= Answer START ========
        out = out.transpose(1, 2).long()
        print(f"Output shape in patch embedding: {out.shape=}")
        # ======= Answer END ========

        return out


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        # n_channels       number of input image channels
        # nout             desired output dimension
        # img_size         width of the square image
        # patch_size       width of the square patch
        # dim              embedding dimension
        # attn_dim         the hidden dimension of the attention layer
        # mlp_dim          the hidden layer dimension of the FFN
        # num_heads        the number of heads in the attention layer
        # num_layers       the number of attention layers.
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        self.pos_E = nn.Embedding(
            (img_size // patch_size) ** 2, dim
        )  # positional embedding matrix

        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))  # learned class embedding
        self.local_position_embedding = copy.deepcopy(self.pos_E)
        self.global_position_embedding = copy.deepcopy(self.cls_token)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.patch_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Projection Head

        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # img          the input image. shape: (B, nin, img_size, img_size)
        # return_attn  whether to return the attention alphas
        #
        # Outputs
        # out          the output of the vision transformer. shape: (B, nout)
        # alphas       the attention weights for all heads and layers. None if return_attn is False, otherwise
        #              shape: (B, num_layers, num_heads, num_patches + 1, num_patches + 1)

        # generate embeddings
        embs = self.patch_embed(img)  # patch embedding
        B, T, _ = embs.shape
        print(f"{embs.shape=}")
        # Positional Encoding
        pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        embs += self.pos_E(pos_ids).long()  # positional embedding
        # todo positional encoding such that there is no limit to sequence size..
        print("VIT\n\n\n")
        print_variance("Embeddings", embs.float())
        cls_token = self.cls_token.expand(len(embs), -1, -1)
        x = torch.cat([cls_token, embs], dim=1)

        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        # print_variance("Output Transformer:", x)
        out = self.head(x)[:, 0]
        print_variance("Projected Output after passing through projection head:", x)
        return out, alphas

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 1,  # Number of global/CLS tokens
    ):
        # should be a mapping... and rotatory

        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)
        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas

# Problem 4: Dialogue GPT

In [ ]:
import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="llm_from_scratch.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

#
# def print(*args):
#     # Convert all inputs to strings and filter out empty values
#     string_elements = [str(arg) for arg in args]
#
#     # If more than 1 item was passed, concatenate them with a space
#     if len(string_elements) > 1:
#         log_message = " ".join(string_elements)
#     elif len(string_elements) == 1:
#         log_message = string_elements[0]
#     else:
#         return  # Skip empty print() calls entirely

# Send the final concatenated message straight to your colored logger
# logger.info(log_message)

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.mps.is_available() else "cpu"
)

In [ ]:
device

In [25]:
import logging
from dataclasses import dataclass
from functools import wraps
from typing import List

import numpy as np
import torch
import torchvision
from torch import nn
from torchvision import transforms
import os

checkpoints = os.makedirs("vit/", exist_ok=True)
checkpoint_dir = "vit"

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


#
# def print(*args):
#     # Convert all inputs to strings and filter out empty values
#     string_elements = [str(arg) for arg in args]
#
#     # If more than 1 item was passed, concatenate them with a space
#     if len(string_elements) > 1:
#         log_message = " ".join(string_elements)
#     elif len(string_elements) == 1:
#         log_message = string_elements[0]
#     else:
#         return  # Skip empty print() calls entirely
#
#     # Send the final concatenated message straight to your colored logger
#     logger.info(log_message)
#


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)


class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


# def averager(func):
#     num = 0
#     total = 0
#
#     def update(val, sz):
#         nonlocal num
#         nonlocal total
#         num += val * sz
#         total += sz
#
#     @wraps(func)
#     def inner():
#         nonlocal num, total
#         update(num / total)
#
#     return inner


def print_variance(name, data):
    neuron_variance = torch.mean(torch.var(data, dim=0))
    print(f"name={name!r}, Variance={neuron_variance.float()}")


class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        super().__init__()
        self.H = num_heads
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.dim = dim // num_heads
        self.qkv = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        # QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        # QKV = QKV.transpose(1, 2)
        Q, K, V = QKV.chunk(3, dim=-1)
        # A = Q @ K.transpose(-2, -1) / self.n_hidden ** 0.5
        # scores = A
        Q = Q.view(batch_size, token_size, self.dim).transpose(
            1, 2
        )  # [B, n_heads, T, head_dim]
        K = K.view(batch_size, token_size, self.dim).transpose(1, 2)
        V = K.view(batch_size, token_size, self.dim).transpose(1, 2)
        if attn_mask is not None:
            # "\n            causal masking future inputs of 0 to negative so that softmax will return 0\n            "
            # scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
            A = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
        else:
            A = F.scaled_dot_product_attention(Q, K, V)

        Z = A @ V
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)
        Z = self.WO(Z)
        return (Z, A)


import torch.nn as nn
import torch.nn.functional as F


class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden  # This is the dimension per head (head_dim)
        self.dim = dim  # Total model embedding dimension

        # Project input to Q, K, V for all heads at once
        self.qkv = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        # Project concatenated head outputs back to model dimension
        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size, token_size, dim = x.shape

        # 1. Project to QKV: [B, T, H * head_dim * 3]
        QKV = self.qkv(x)

        # 2. Reshape and split into Q, K, V
        # First expand to separate heads and the 3 projections: [B, T, H, 3, head_dim]
        QKV = QKV.view(batch_size, token_size, self.H, 3, self.n_hidden)
        # Permute to move heads and projection type forward: [3, B, H, T, head_dim]
        QKV = QKV.permute(3, 0, 2, 1, 4)
        # Extract individual Q, K, V tensors: each is [B, H, T, head_dim]
        Q, K, V = QKV[0], QKV[1], QKV[2]

        # 3. Compute Attention using PyTorch built-in function
        # F.scaled_dot_product_attention performs QK^T scaling, softmax, AND multiplies by V
        # It expects shape [B, H, T, head_dim] and outputs [B, H, T, head_dim]
        if attn_mask is not None:
            # Note: If passing an explicit custom mask, set is_causal=False
            A = F.scaled_dot_product_attention(
                Q, K, V, attn_mask=attn_mask, is_causal=False
            )
        else:
            A = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

        # 4. Concatenate heads back together
        # Move sequence length back to dim 1: [B, T, H, head_dim]
        A = (
            A.transpose(1, 2)
            .contiguous()
            .view(batch_size, token_size, self.H * self.n_hidden)
        )
        # Flatten heads and hidden dimensions: [B, T, H * head_dim]
        # A = out.view(batch_size, token_size, self.H * self.n_hidden)

        # 5. Output projection: [B, T, dim]
        Z = self.WO(A)

        # Note: F.scaled_dot_product_attention does not return raw attention weights.
        # We return None for the weights, or you can return out.
        return Z, A


import torch
from torch import nn

#
# class MultiHeadedAttention(nn.Module):
#     def __init__(self, dim: int, n_hidden: int, num_heads: int):
#         super().__init__()
#         self.dim = dim
#         self.num_heads = num_heads
#         self.n_hidden = n_hidden
#         self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
#         self.W0 = nn.Linear(num_heads * n_hidden, dim)
#
#     def forward(self, x: torch.Tensor, attn_mask=None) -> tuple[torch.Tensor, torch.Tensor]:
#         B, T, dim = x.shape
#         qkv = self.qkv_projection(x)
#         qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
#         q, k, v = qkv.chunk(3, dim=-1)
#         scores = torch.matmul(q, k.transpose(-2, -1)) / self.n_hidden ** 0.5
#         if attn_mask is not None:
#             A = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))  # causal mask
#         attn_alphas = torch.softmax(scores, dim=-1)
#         A = F.scaled_dot_product_attention(q, k, v, is_causal=True)
#         context = attn_alphas @ v
#         A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)
#         Z = self.W0(A)
#         return (Z, attn_alphas)


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiAttentionHead(dim, attn_dim, num_heads)
        # self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )
        # self.norm2 = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, attn_mask=False
    ) -> tuple[torch.Tensor, torch.Tensor]:
        print_variance("Input", x)
        Z, A = self.attn(x, attn_mask=attn_mask)
        print_variance("after first attention block output", Z)
        x = Z + x
        print_variance("after residual 1 adding output to Z", x)
        ffn_out = self.ffn(x)
        x = ffn_out + x
        print_variance("after Applying FeedForward output with residual", x)
        print("Residual Block\n\n")
        return (x, A)


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self, x: torch.Tensor, attn_mask=False, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []
        Z = None
        A = []
        print(f"\n\n-------------------------Transformer: {x.shape}")
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)
            if return_attn is not None:
                A.append(alphas)
        if return_attn:
            return (x, torch.stack(A, dim=1))
        else:
            return (x, None)


class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.flattened_patch = (img_size // patch_size) ** 2
        self.patch_embedding = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.nout = nout

    def forward(self, x: torch.Tensor):
        """TODO: Implement the patch embedding. You want to split up the image into square patches of the given patch size. Then each patch_size x patch_size square should be linearly projected into an embedding of size nout. Hint: Take a look at nn.Conv2d. How can this be used to perform the patch embedding?"""
        out = self.patch_embedding(x)
        out = out.flatten(2)
        out = out.transpose(1, 2)
        logger.info(f"Output shape in patch embedding: {out.shape=}")
        return out


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        global_tokens: int,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        # Todo add global embeddings so that even token window does not matter Then we can learn very fine grained details..

        self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)
        # cls token-> start of patch or token sequence
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        # nn.init.trunc_normal_()
        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        # Projection HEad
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        embs = self.patch_embed(img)
        B, T, _ = embs.shape
        logger.info(f"embs.shape={embs.shape!r}")
        # Number of learnt position windows up to T tokens Local embeddings
        pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        embs = embs + self.pos_E(pos_ids)
        # print_variance("Embeddings", embs.float())
        cls_token = self.cls_token.expand(len(embs), -1, -1)
        x = torch.cat([cls_token, embs], dim=1)
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        # print_variance("Output", x)
        out = self.head(x)[:, 0]
        # print_variance("Projected Output", x)
        return (out, alphas)


# evaluate the model
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()


result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset, batch_size=256, shuffle=True, num_workers=2
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset, batch_size=256, shuffle=False, num_workers=2
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
    )
    # result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        best_models = "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
        best_save_path = os.path.join(checkpoint_dir, best_models)
        model_path = "Vision_transformer_" + str(epoch + 1) + ".pt"
        save_path = os.path.join(checkpoint_dir, model_path)

        # todo add an early stopping criteria and restore best weights

        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:

            torch.save(model.state_dict(), best_save_path)
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            print("Finished Training")

        else:
            torch.save(model.state_dict(), save_path)
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        clean_memory_cache()
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
        print("Finished Training")

In [28]:
from typing import Optional, Tuple, Any, List
import torch
import torch.nn as nn
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy
from torch.utils.data import DataLoader, Subset

torch.autograd.set_detect_anomaly(True)
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
import os

num_workers = min(0, os.cpu_count())
import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.w_o = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape

        # Shape: (B, T, 3 * num_heads * n_hidden) -> (B, num_heads, T, 3 * n_hidden)
        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        q, k, v = qkv.chunk(3, dim=-1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)
            scores = scores.masked_fill(attn_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        output = self.w_o(context)
        return output, attn_weights


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)

        # LayerNorm applied inside the FFN sequence only
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(x, attn_mask=attn_mask)
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(x)
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        collected_attns = []

        for layer in self.layers:
            x, alphas = layer(x, attn_mask=attn_mask)
            if return_attn:
                collected_attns.append(alphas)

        if return_attn:
            return x, torch.stack(collected_attns, dim=1)
        return x, None


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert (
            img_size % patch_size == 0
        ), "Image dimensions must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Output: (B, num_patches, nout) in float
        return self.proj(x).flatten(2).transpose(1, 2)


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 1,  # Number of global/CLS tokens
    ):
        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)

        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas


# evaluate the model
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()


result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)
    print(model)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
    )
    # result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        best_models = "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
        best_save_path = os.path.join(checkpoint_dir, best_models)
        model_path = "Vision_transformer_" + str(epoch + 1) + ".pt"
        save_path = os.path.join(checkpoint_dir, model_path)

        # todo add an early stopping criteria and restore best weights

        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:

            torch.save(model.state_dict(), best_save_path)
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            print("Finished Training")

        else:
            torch.save(model.state_dict(), save_path)
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        clean_memory_cache()
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
    print("Finished Training")

In [29]:
num_workers

0

In [30]:
os.cpu_count()

2

In [31]:
device

'cuda'

In [32]:
model = VisionTransformer(
    n_channels=3,
    nout=10,
    img_size=32,
    patch_size=4,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)
print(model)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
  )
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (w_o): Linear(in_features=192, out_features=128, bias=True)
        )
        (ffn): Sequential(
          (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=128, out_features=128, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [20]:
# os.chdir('../')

In [21]:
# !rm -rf cifar-10-batches-py

In [33]:
main()  # 13 m 13 seconds on colab

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
  )
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (w_o): Linear(in_features=192, out_features=128, bias=True)
        )
        (ffn): Sequential(
          (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=128, out_features=128, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=10, bias=True)
  )
)


Training at 0: 100%|██████████| 196/196 [01:05<00:00,  3.00it/s]


Train Epoch: 0, Loss: 1.745776955909729, Acc: 0.352200000038147
Val Epoch: 0, Loss: 1.565373733139038, Acc: 0.4314
34.45 MB
Val Epoch: 1, Loss: 1.565373733139038, Acc: 0.4314


Training at 1: 100%|██████████| 196/196 [01:05<00:00,  2.99it/s]


Train Epoch: 1, Loss: 1.3383928649520873, Acc: 0.51622
Val Epoch: 1, Loss: 1.2268987022399902, Acc: 0.5525
34.45 MB
Val Epoch: 2, Loss: 1.2268987022399902, Acc: 0.5525


...

Train Epoch: 7, Loss: 0.5694896374130249, Acc: 0.8012200000190735
Val Epoch: 7, Loss: 0.8357551581382752, Acc: 0.7154
34.45 MB
Val Epoch: 8, Loss: 0.8357551581382752, Acc: 0.7154


Training at 8: 100%|██████████| 196/196 [01:06<00:00,  2.94it/s]


Train Epoch: 8, Loss: 0.48706892183303835, Acc: 0.8330800000572205
Val Epoch: 8, Loss: 0.817731090927124, Acc: 0.7267
34.45 MB
Val Epoch: 9, Loss: 0.817731090927124, Acc: 0.7267


Training at 9: 100%|██████████| 196/196 [01:06<00:00,  2.94it/s]


Train Epoch: 9, Loss: 0.435453254699707, Acc: 0.853860000038147
Val Epoch: 9, Loss: 0.8158617504119873, Acc: 0.7311
34.45 MB
Val Epoch: 10, Loss: 0.8158617504119873, Acc: 0.7311
Finished Training


In [ ]:
model = Transformer(dim=128, attn_dim=64, mlp_dim=128, num_heads=3, num_layers=6)

In [ ]:
model

In [34]:
import pandas as pd

df = pd.DataFrame.from_dict(
    {
        "train_acc": result.train_accs,
        "train_loss": result.train_losses,
        "val_acc": result.val_accs,
        "val_loss": result.val_losses,
    }
)
df

,train_acc,train_loss,val_acc,val_loss
0,0.35220,1.745777,0.4314,1.565374
1,0.51622,1.338393,0.5525,1.226899
2,0.59786,1.120903,0.6104,1.093584
3,0.65182,0.974819,0.6419,1.008526
4,0.69004,0.867641,0.6779,0.914095
5,0.73266,0.758592,0.6958,0.868042
6,0.76708,0.657041,0.7094,0.846851
7,0.80122,0.569490,0.7154,0.835755
8,0.83308,0.487069,0.7267,0.817731
9,0.85386,0.435453,0.7311,0.815862


In [35]:
df.to_csv("results.csv")

In [36]:
!pip install wget

In [ ]:
from pprint import pprint

In [38]:
# import os
#
import wget

#
if not os.path.exists("shakespeare.txt"):
    wget.download(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    )

In [39]:
with open("input.txt", "r") as f:
    raw_text = f.read()

In [40]:
all_dialogues = raw_text.split("\n\n")

In [41]:
from pprint import pprint

pprint(all_dialogues[:10])

['First Citizen:\nBefore we proceed any further, hear me speak.',
 'All:\nSpeak, speak.',
 'First Citizen:\nYou are all resolved rather to die than to famish?',
 'All:\nResolved. resolved.',
 'First Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.',
 "All:\nWe know't, we know't.",
 'First Citizen:\n'
 "Let us kill him, and we'll have corn at our own price.\n"
 "Is't a verdict?",
 "All:\nNo more talking on't; let it be done: away, away!",
 'Second Citizen:\nOne word, good citizens.',
 'First Citizen:\n'
 'We are accounted poor citizens, the patricians good.\n'
 'What authority surfeits on would relieve us: if they\n'
 'would yield us but the superfluity, while it were\n'
 'wholesome, we might guess they relieved us humanely;\n'
 'but they think we are too dear: the leanness that\n'
 'afflicts us, the object of our misery, is as an\n'
 'inventory to particularise their abundance; our\n'
 'sufferance is a gain to them Let us revenge this with\n'
 'our pikes, ere we be

In [42]:
for dialogue in all_dialogues[:10]:
    print(dialogue)

First Citizen:
Before we proceed any further, hear me speak.
All:
Speak, speak.
First Citizen:
You are all resolved rather to die than to famish?
All:
Resolved. resolved.
First Citizen:
First, you know Caius Marcius is chief enemy to the people.
All:
We know't, we know't.
First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?
All:
No more talking on't; let it be done: away, away!
Second Citizen:
One word, good citizens.
First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.


In [43]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Part 4.A

In [44]:
def tokenize(s):
    return word_tokenize(s)

In [45]:
print(word_tokenize(all_dialogues[0]))

['First', 'Citizen', ':', 'Before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.']


In [46]:
print(all_dialogues[0])

First Citizen:
Before we proceed any further, hear me speak.


In [47]:
def tokenize(s):
    return word_tokenize(s)

In [48]:
for dialogue in all_dialogues[:10]:
    print(np.unique(word_tokenize(dialogue)))

[',' '.' ':' 'Before' 'Citizen' 'First' 'any' 'further' 'hear' 'me'
 'proceed' 'speak' 'we']
[',' '.' ':' 'All' 'Speak' 'speak']
[':' '?' 'Citizen' 'First' 'You' 'all' 'are' 'die' 'famish' 'rather'
 'resolved' 'than' 'to']
['.' ':' 'All' 'Resolved' 'resolved']
[',' '.' ':' 'Caius' 'Citizen' 'First' 'Marcius' 'chief' 'enemy' 'is'
 'know' 'people' 'the' 'to' 'you']
[',' '.' ':' 'All' 'We' "know't" 'we']
["'ll" ',' '.' ':' '?' 'Citizen' 'First' "Is't" 'Let' 'a' 'and' 'at'
 'corn' 'have' 'him' 'kill' 'our' 'own' 'price' 'us' 'verdict' 'we']
['!' ',' ':' ';' 'All' 'No' 'away' 'be' 'done' 'it' 'let' 'more' "n't" 'o'
 'talking']
[',' '.' ':' 'Citizen' 'One' 'Second' 'citizens' 'good' 'word']
[',' '.' ':' ';' 'Citizen' 'First' 'I' 'Let' 'We' 'What' 'a' 'abundance'
 'accounted' 'afflicts' 'an' 'are' 'as' 'authority' 'become' 'bread' 'but'
 'citizens' 'dear' 'ere' 'for' 'gain' 'gods' 'good' 'guess' 'humanely'
 'hunger' 'if' 'in' 'inventory' 'is' 'it' 'know' 'leanness' 'might'
 'misery' 'not' 'ob

In [49]:
x = tokenize(raw_text)

In [50]:
x = torch.tensor([0, 1, 2])
for k in x:
    print(k)

tensor(0)
tensor(1)
tensor(2)


In [51]:
def tokenize(s):
    return word_tokenize(s)


class MyTokenizer:
    def __init__(self, raw_text: str):
        # raw_text     contains the text from which we will build our vocabulary

        self.start = "<START>"  # token that starts every example
        self.pad = "<PAD>"  # token used to pad examples to the same length
        self.unk = "<UNK>"  # token used if encountering a word not in our vocabulary

        vocab = np.unique(tokenize(raw_text))
        vocab = np.concatenate([np.array([self.start, self.pad, self.unk]), vocab])

        self.vocab = vocab  # array of tokens in order
        self.tok_to_id = {w: i for i, w in enumerate(vocab)}  # mapping of token to ID
        self.id_to_token = {i: w for i, w in enumerate(vocab)}
        self.vocab_size = len(self.vocab)  # size of vocabulary

    def __len__(self):
        return self.vocab_size

    def encode(self, s: str) -> torch.Tensor:
        # s           input string
        #
        # Output
        # id_tensor   a tensor of token ids, starting with the start token.t

        id_tensor = torch.from_numpy(
            np.array(
                [self.tok_to_id[self.start]]
                + [self.tok_to_id[w] for w in tokenize(s) if w in self.tok_to_id],
                dtype=np.int32,
            )
        )

        # TODO: tokenize the input using word_tokenize. Return a tensor  of the token ids, starting with the token id for the start token.
        # ============ ANSWER START ===========
        # encoded_string = tokenize(s)
        # token_ids =
        # token_ids.append(self.tok_to_id[self.start])
        # token_ids.extend(
        #     [self.tok_to_id[w] for w in encoded_string if w in self.tok_to_id.keys()]
        # )
        # id_tensor = np.array(token_ids)

        # id_tensor = torch.from_numpy(id_tensor)
        # ============ ANSWER END =============

        return id_tensor

    def decode(self, toks: torch.Tensor) -> str:
        # toks         a list of token ids
        #
        # Output
        # decoded_str  the token ids decoded back into a string (join with a space)

        # TODO: convert the token ids back to the actual corresponding words.
        # Join the tokens with a space and return the full string
        # ============ ANSWER START ===========
        return " ".join(
            [
                self.id_to_token[int(token)]
                for token in toks
                if token in self.tok_to_id.values()
            ]
        ).rstrip()

        # ============ ANSWER END =============

        # return decoded_str

    def pad_examples(self, tok_list: List[torch.Tensor]) -> torch.Tensor:
        # Pads the tensors to the right with the pad token so that they are the same length.
        #
        # tok_list       a list of tensors containing token ids (maybe of different lengths)
        #
        # Output
        # padded_tokens  shape: (len(tok_list), max length within tok_list)
        return torch.nn.utils.rnn.pad_sequence(
            tok_list, batch_first=True, padding_value=self.tok_to_id[self.pad]
        )


tok = MyTokenizer(raw_text)

In [52]:
len(tok)

14213

In [53]:
all_dialogues[:2]

['First Citizen:\nBefore we proceed any further, hear me speak.',
 'All:\nSpeak, speak.']

In [54]:
tok.tok_to_id[tok.start]

0

In [55]:
tok.id_to_token[1986]

np.str_('Ovid')

In [56]:
tok.id_to_token[0]

np.str_('<START>')

In [57]:
np.array([str(tok.id_to_token[0]), "Hi Deven"])

array(['<START>', 'Hi Deven'], dtype='<U8')

In [58]:
# tokenizer test cases
input_string = "KING RICHARD III:\nSay that I did all this for love of her. bluye"
enc = tok.encode(input_string)
print(enc)

# for x in enc:
#     print(x, type(x), x in tok.tok_to_id.values(), x in tok.id_to_token.values())
dec = tok.decode(enc)
print(dec)
# print("<START> KING RICHARD III : Say that I did all this for love of her .")
assert dec == "<START> KING RICHARD III : Say that I did all this for love of her ."

tensor([    0,  1593,  2182,  1481,   223,  2343, 12742,  1476,  5704,  3319,
        12795,  6848,  8727,  9608,  7655,   221], dtype=torch.int32)
<START> KING RICHARD III : Say that I did all this for love of her .


In [ ]:
# enc = tok.encode(all_dialogues[])

# Part 4.B

### Debug

In [3]:
import os

# Sets the high watermark ratio to 0.0, completely disabling the upper memory allocation limits
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"


class DialogueDataset:
    def __init__(self, tokenizer: MyTokenizer, lines: List[str], max_N: int):
        # tokenizer    an instance of MyTokenizer
        # lines        a list of strings. each element in an example in the dataset
        # max_N        the maximum number of tokens allowed per example. More than this will be truncated
        self.lines = lines
        self.tokenizer = tokenizer
        self.max_N = max_N

    def __len__(self) -> int:
        return len(self.lines)

    # def __iter__(self):
    #     for line in self.lines:
    #         yield self.tokenizer.encode(line)[: self.max_N]

    def __getitem__(self, idx: int) -> torch.Tensor:
        # returns the example at int encoded by the tokenizer
        # truncates the example if it is more than max_N tokens
        return self.tokenizer.encode(self.lines[idx])[: self.max_N]

    # def __getitems__(self,indices:int):
    #     return [self.__getitem__(idx) for idx in indices]


ds = DialogueDataset(tok, all_dialogues, max_N=200)

NameError: name 'tok' is not defined

In [1]:
def collate_fn(examples: List[torch.Tensor]):
    """
    # examples        a batch of tensors containing token ids (maybe of different lengths)
    # Outputs a dictionary containing
    #   input_ids     a single tensor with all of the examples padded (from the right) to the max
    #                 length within the batch. shape:(B, max length within examples)
    #   input_mask    a tensor indicating which tokens are padding and should be ignored. 0 if padding
    #                 and 1 if not. shape: (B, max length within examples)
    """
    new_input_ids = tok.pad_examples(examples)
    attn_mask = torch.ones(new_input_ids.shape)  # 1s should not be ignored

    # causal attention mask
    attn_mask[new_input_ids == tok.tok_to_id[tok.pad]] = (
        0  # should be ignored if it is a padded
    )
    return {"input_ids": tok.pad_examples(examples), "input_mask": attn_mask}

In [2]:
first = ds[2]
second = ds[1]
# token_list = torch.tensor([first,second])
tokens = [first, second]
torch.nn.utils.rnn.pad_sequence(tokens, padding_value=tok.tok_to_id[tok.pad])

NameError: name 'ds' is not defined

In [62]:
d = collate_fn(tokens)

In [63]:
d["input_ids"]

tensor([[    0,  1151,   708,   223,  3050,  3506,  3319, 10983, 10724, 12921,
          5706, 12733, 12921,  6507,   225],
        [    0,   323,   223,  2517,   219, 12008,   221,     1,     1,     1,
             1,     1,     1,     1,     1]], dtype=torch.int32)

In [64]:
d["input_mask"]

tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [65]:
clean_memory_cache()

21.20 MB


In [66]:
from torch.utils.data import DataLoader, Subset

In [67]:
training_dl = torch.utils.data.DataLoader(
    ds, batch_size=64, collate_fn=collate_fn
)  # collate function is just a dictionary..

In [68]:
input_id, input_mask = next(iter(training_dl)).items()

In [69]:
a, b = (next(iter(training_dl))).items()

In [70]:
x1, x2 = a

In [71]:
print(x1)
x2

input_ids


tensor([[   0, 1151,  708,  ...,    1,    1,    1],
        [   0,  323,  223,  ...,    1,    1,    1],
        [   0, 1151,  708,  ...,    1,    1,    1],
        ...,
        [   0, 1733,  223,  ...,    1,    1,    1],
        [   0, 1151, 2385,  ...,    1,    1,    1],
        [   0, 1733,  223,  ...,    1,    1,    1]], dtype=torch.int32)

In [72]:
# take a look at an example of an element from the training dataloader
from IPython.display import display

for batch in training_dl:
    print(batch.keys())
    print(batch["input_ids"])
    break

dict_keys(['input_ids', 'input_mask'])
tensor([[   0, 1151,  708,  ...,    1,    1,    1],
        [   0,  323,  223,  ...,    1,    1,    1],
        [   0, 1151,  708,  ...,    1,    1,    1],
        ...,
        [   0, 1733,  223,  ...,    1,    1,    1],
        [   0, 1151, 2385,  ...,    1,    1,    1],
        [   0, 1733,  223,  ...,    1,    1,    1]], dtype=torch.int32)


## Part 4.C

In [34]:
embs = torch.ones((32, 100, 128))
B, T, _ = embs.shape
pos_ids = torch.arange(T).expand(B, -1)
print(f"{pos_ids.shape=}")
pos_E = nn.Embedding(200, 128)
print(pos_E)
x = pos_E(pos_ids)

pos_ids.shape=torch.Size([32, 100])
Embedding(200, 128)


In [35]:
print(f"{B=},{T=}")

B=32,T=100


In [36]:
pos_E

Embedding(200, 128)

In [37]:
a = torch.arange(T).expand(size=(B, -1))
embedding_layer = nn.Embedding(100, 128)
x = embedding_layer(a)

In [38]:
x = x.detach()

In [39]:
x.shape

torch.Size([32, 100, 128])

In [40]:
torch.tensor(10).repeat(2, 1, 2).shape

torch.Size([2, 1, 2])

In [41]:
torch.ones(B, B).unsqueeze(dim=0).repeat(1, 1, 1).shape

torch.Size([1, 32, 32])

In [42]:
x.shape

torch.Size([32, 100, 128])

In [43]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        # Todo add global embeddings so that even token window does not matter Then we can learn very fine grained details..

        self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)
        # cls token-> start of patch or token sequence
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        # Projection Head
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        embs = self.patch_embed(img)
        B, T, _ = embs.shape
        logger.info(f"embs.shape={embs.shape!r}")
        # Number of learnt position windows up to T tokens Local embeddings
        pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        embs = embs + self.pos_E(pos_ids).long()  # Meaning?
        print_variance("Embeddings", embs.float())
        cls_token = self.cls_token.expand(len(embs), -1, -1)
        x = torch.cat([cls_token, embs], dim=1)
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        # print_variance("Output", x)
        out = self.head(x)[:, 0]
        print_variance("Projected Output", x)
        return (out, alphas)

In [44]:
assert (torch.tril(torch.ones(B, B)).unsqueeze(dim=0).repeat(100, 1, 1)).shape == (
    T,
    B,
    B,
)

In [73]:
class DialogueGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_N: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        # vocab_size       size of the vocabulary
        # max_N            maximum number of tokens allowed to appear in 1 example
        # dim              embedding dimension
        # attn_dim         the hidden dimension of the attention layer
        # mlp_dim          the hidden layer dimension of the FFN
        # num_heads        the number of heads in the attention layer
        # num_layers       the number of attention layers.

        super().__init__()
        """
        • Given the token ids, retrieve the corresponding token embeddings. Add to this a learned positional embedding.
        • Generate a causal attention mask. Remember that for GPT, every token only depends on itself and the tokens before it
        • Pass the embeddings and the attention mask to the transformer and the language model head. Output logits of size (T ×V) where T is the number of tokens and V is the vocabulary size. This step is implemented for you
        """
        # TODO: set up the token embedding and positional embeddings
        #       Hint, use nn.Embedding
        # Already Padded to keep sequence length same
        # Next use Graph Neural Network
        # token_embeddings

        self.token_embeddings = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=dim
        )

        self.pos_embeddings = nn.Embedding(num_embeddings=max_N, embedding_dim=dim)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        # Projection Head
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, vocab_size))

    def forward(
        self, input_ids: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # input_ids     a batch of input ids (right padded). shape: (B x T)
        # return_attn   whether to return the attention weights
        #
        # Output
        # out           the logit vector (B x T x V)
        # alphas        the attention weights if return_attn is True. Otherwise None shape: (B, num_layers, num_heads, T, T)
        """

        Args:
            input_ids: Batch of input ids right padded shape: (BxT)
            return_attn: whether to return the attention weights
        • Generate a causal attention mask. Remember that for Generative  Pretrained Transformer, every token only depends on itself and the tokens before it
        • Pass the embeddings and the attention mask to the transformer and the language model head. Output logits of size (T ×V) where T is the number of tokens and V is the vocabulary size. This step is implemented for you
        Returns:

        """
        B, T = input_ids.shape
        pos_ids = torch.arange(0, T, dtype=torch.long, device=device).unsqueeze(
            0
        )  # Shape: (1, T)
        embs = self.token_embeddings(input_ids) + self.pos_embeddings(pos_ids)
        # TODO: retrieve the token embeddings for the input_ids.
        #       Add to the token embeddings the positional embeddings.
        #       Store the combined embedding in embs

        # TODO: Create the causal attention mask, which should be of size (B, T, T)
        #       Remember that the causal attention mask is lower triangular (all tokens only
        #       depend on themselves and the tokens before them).   Store the mask in causal_attn_mask
        # Hint: check out torch.tril creates a causal mask where each token can only attend to previous tokens and itself
        causal_attn_mask = (
            torch.tril(torch.ones(T, T)).unsqueeze(0).repeat(B, 1, 1)
        ).to(
            device
        )  # Shape: (B, T, T)
        # ============ ANSWER START ============

        # ============ ANSWER END ==============

        x, alphas = self.transformer(
            embs, attn_mask=causal_attn_mask, return_attn=return_attn
        )
        out = self.head(x)
        return out, alphas

    def generate(self, input_ids, num_tokens):
        # you can assume batch size 1
        # greedy generation
        with torch.no_grad():
            for i in range(num_tokens):
                out, _ = self.forward(input_ids)
                new_token = torch.argmax(out[:, [-1]], -1)
                input_ids = torch.cat([input_ids, new_token], dim=1)
        return input_ids

In [74]:
device

'cuda'

In [75]:
model = DialogueGPT(
    vocab_size=len(tok),
    max_N=200,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)

In [76]:
model

DialogueGPT(
  (token_embeddings): Embedding(14213, 128)
  (pos_embeddings): Embedding(200, 128)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (w_o): Linear(in_features=192, out_features=128, bias=True)
        )
        (ffn): Sequential(
          (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=128, out_features=128, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=14213, bias=True)
  )
)

In [77]:
len(tok)

14213

In [78]:
batch = next(iter(training_dl))

In [79]:
batch

{'input_ids': tensor([[   0, 1151,  708,  ...,    1,    1,    1],
         [   0,  323,  223,  ...,    1,    1,    1],
         [   0, 1151,  708,  ...,    1,    1,    1],
         ...,
         [   0, 1733,  223,  ...,    1,    1,    1],
         [   0, 1151, 2385,  ...,    1,    1,    1],
         [   0, 1733,  223,  ...,    1,    1,    1]], dtype=torch.int32),
 'input_mask': tensor([[1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         ...,
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.]])}

In [80]:
batch.keys()

dict_keys(['input_ids', 'input_mask'])

In [81]:
model = model.to(device)
x = batch["input_ids"]
mask = batch["input_mask"]
x = x.to(device)
model(x)

(tensor([[[ 8.4802e-02,  4.4081e-01,  4.2037e-01,  ...,  2.7880e-01,
           -7.5163e-02,  1.4744e+00],
          [ 1.0958e+00, -2.4750e-01, -8.2184e-02,  ..., -6.3016e-02,
           -3.3323e-01,  5.0624e-01],
          [ 7.6456e-01,  9.1840e-01, -1.3560e-01,  ...,  2.1659e-01,
           -1.7583e-01,  5.3149e-01],
          ...,
          [-3.8910e-02, -1.8953e-02,  2.5576e-01,  ..., -3.1680e-01,
            7.0983e-01,  2.4104e-03],
          [-2.6515e-01, -1.7013e-01, -1.3726e-02,  ...,  8.5608e-01,
            4.1702e-01, -2.2402e-01],
          [ 1.0662e-01, -1.0876e-01, -6.3563e-02,  ..., -4.1420e-01,
            7.3558e-01,  1.8046e-01]],
 
         [[ 8.4802e-02,  4.4081e-01,  4.2037e-01,  ...,  2.7880e-01,
           -7.5163e-02,  1.4744e+00],
          [ 1.1903e+00, -2.4606e-01,  7.1517e-01,  ..., -6.4459e-01,
            6.5822e-01,  3.9161e-01],
          [ 7.5936e-01,  4.5247e-02, -3.9294e-01,  ...,  4.6345e-01,
            4.7930e-01,  5.1523e-01],
          ...,
    

In [82]:
x

tensor([[   0, 1151,  708,  ...,    1,    1,    1],
        [   0,  323,  223,  ...,    1,    1,    1],
        [   0, 1151,  708,  ...,    1,    1,    1],
        ...,
        [   0, 1733,  223,  ...,    1,    1,    1],
        [   0, 1151, 2385,  ...,    1,    1,    1],
        [   0, 1733,  223,  ...,    1,    1,    1]], device='cuda:0',
       dtype=torch.int32)

In [55]:
mask

tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.]])

In [56]:
mask = mask.to(device)
x * mask

tensor([[   0.,  950.,  505.,  ...,    0.,    0.,    0.],
        [   0.,  118.,   18.,  ...,    0.,    0.,    0.],
        [   0.,  950.,  505.,  ...,    0.,    0.,    0.],
        ...,
        [   0., 1536.,   18.,  ...,    0.,    0.,    0.],
        [   0.,  950., 2192.,  ...,    0.,    0.,    0.],
        [   0., 1536.,   18.,  ...,    0.,    0.,    0.]], device='mps:0')

## Part 4.D

In [57]:
a = torch.tensor([1, 2, 3])
a = a[:-1]
a

tensor([1, 2])

In [83]:
clean_memory_cache()

1353.02 MB


In [84]:
class DialogueLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(reduction="none")

    def forward(
        self, logits: torch.Tensor, input_ids: torch.Tensor, inp_mask: torch.Tensor
    ):
        """
        # logits      the logits produced by DialogueGPT. shape: (B x T x V)
        # input_ids   the token ids. shape: (B x T)
        # inp_mask    a 0/1 mask of which tokens are padding tokens and should be ignored. shape: (B x T)

        TODO: Implement the language model loss. For logits[i], we want to supervise the i+1 token_id with the cross entropy loss. We thus will not supervise the start token (input_ids[0]) or use the last logit vector (logits[-1]). Return the average of the losses for each token in the batch, making sure to ignore tokens corresponding to the padding (use inp_mask).
        """
        loss = 0

        # start_token = input_ids[0]
        relevant_logits = logits[:, :-1, :]
        # print(f"{logits.shape=},{relevant_logits.shape=}")
        relevant_tokens = input_ids[:, 1:]
        shift_mask = inp_mask[:, 1:]
        relevant_logits = relevant_logits.permute(0, 2, 1)
        # print(f"{relevant_logits.shape=}")

        loss = self.criterion(relevant_logits, relevant_tokens)
        shift_mask = shift_mask.to(dtype=loss.dtype, device=loss.device)
        loss *= shift_mask

        # ============ ANSWER START ============

        # ============ ANSWER END ==============
        return torch.sum(loss) / torch.clamp(torch.sum(shift_mask), min=1e-9)

In [99]:
import torch
import torch.nn as nn


class DialogueLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Explicitly ignore padding index if your mask allows it,
        # but since we manual-mask below, reduction="none" is perfect.
        self.criterion = nn.CrossEntropyLoss(reduction="none")

    def forward(
        self, logits: torch.Tensor, input_ids: torch.Tensor, inp_mask: torch.Tensor
    ):
        # Shift logits and targets for next-token prediction
        relevant_logits = logits[:, :-1, :]  # Shape: (B, T-1, V)
        relevant_tokens = input_ids[:, 1:]  # Shape: (B, T-1)
        shift_mask = inp_mask[:, 1:]  # Shape: (B, T-1)

        # PyTorch CrossEntropyLoss expects vocabulary dimension as the second dimension: (B, V, T-1)
        relevant_logits = relevant_logits.permute(0, 2, 1)

        # Compute token-wise loss
        loss = self.criterion(relevant_logits, relevant_tokens)

        # FIX: Ensure mask matches the loss tensor's data type AND device exactly
        shift_mask = shift_mask.to(dtype=loss.dtype, device=loss.device)

        # Apply mask to nullify losses on padding tokens
        loss = loss * shift_mask
        # print(f"{loss.detach().dtype}")

        # Avoid division by zero smoothly using clamp
        ans = torch.sum(loss) / torch.clamp(torch.sum(shift_mask), min=1e-9)
        print(f"{ans.dtype=}")
        return ans

In [100]:
loss = nn.CrossEntropyLoss(reduction="none")
# Outputs a tensor of the same size as the batch (or spatial dimensions for tasks like segmentation), giving you full control over how to weight, filter, or apply custom masks to individual samples

In [101]:
inp = torch.randn(2, 2, requires_grad=True)
target = torch.randn(2, 2)

In [102]:
all_dialogues[0]

'First Citizen:\nBefore we proceed any further, hear me speak.'

In [103]:
target.sum(axis=1)

tensor([ 0.6776, -1.8141])

In [104]:
loss(inp, target)

tensor([ 0.2813, -1.1052], grad_fn=<NegBackward0>)

In [105]:
logits = torch.randn(3, 5, requires_grad=True)
targets = torch.randn(3, 5)
x = loss(logits, targets)

In [106]:
x.detach().dtype

torch.float32

In [107]:
logits

tensor([[-1.4744,  2.5705,  1.4206, -0.1559,  1.2842],
        [-0.9113, -1.4445,  0.6720,  1.1666, -0.1936],
        [-0.1044, -1.6210,  0.7014, -1.1856,  1.6944]], requires_grad=True)

In [92]:
target

tensor([[-1.0039, -0.7822],
        [ 0.3289,  0.1875]])

## Part 4.F

In [112]:
import torch.optim as optim

model = DialogueGPT(
    vocab_size=tok.vocab_size,
    max_N=200,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)
criterion = DialogueLoss()

NUM_EPOCHS = 80

optimizer = optim.AdamW(
    model.parameters(), lr=0.0001, weight_decay=0
)  # implement in homework
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [114]:
# model()

In [127]:
# training_dl.dataset

In [119]:
dl = Subset(training_dl.dataset, torch.arange(3))

In [120]:
from torch.utils.data import DataLoader

mini_dl = DataLoader(dl, batch_size=training_dl.batch_size)

In [94]:
len(training_dl)

113

In [95]:
# next(iter(mini_dl))

In [115]:
# Time estimate: around 30 minutes on T4 GPU
# Training
import tqdm

for epoch in range(NUM_EPOCHS):  # loop over the dataset multiple times
    loss_meter = AverageMeter()
    for inp_dict in tqdm.tqdm(training_dl):
        # get the inputs; data is a list of [inputs, labels]
        inp_ids, inp_mask = inp_dict["input_ids"], inp_dict["input_mask"]

        inp_ids = inp_ids.to(device).long()
        inp_mask = inp_mask.to(device)
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs, _ = model(input_ids=inp_ids)
        # print(f"{outputs.shape=}")
        loss = criterion(outputs, inp_ids, inp_mask)
        loss_meter.update(loss.item(), len(inp_dict["input_ids"]))
        loss.backward()
        optimizer.step()
    scheduler.step()

    # print example
    inp = tok.encode("").unsqueeze(0).to(device)
    print(tok.decode(model.generate(inp, 10)[0].cpu()))

    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate():0.4f}, LR: {scheduler.get_last_lr()[0]}"
    )

  0%|          | 0/113 [00:00<?, ?it/s]

outputs.shape=torch.Size([64, 200, 14213])
torch.float32
ans.dtype=torch.float32


  2%|▏         | 2/113 [00:00<00:46,  2.38it/s]

outputs.shape=torch.Size([64, 192, 14213])
torch.float32
ans.dtype=torch.float32


  3%|▎         | 3/113 [00:01<00:44,  2.50it/s]

...

outputs.shape=torch.Size([64, 161, 14213])
torch.float32
ans.dtype=torch.float32


 99%|█████████▉| 112/113 [00:45<00:00,  3.20it/s]

outputs.shape=torch.Size([64, 36, 14213])
torch.float32
ans.dtype=torch.float32


100%|██████████| 113/113 [00:45<00:00,  2.49it/s]

outputs.shape=torch.Size([54, 97, 14213])
torch.float32
ans.dtype=torch.float32
<START> KING RICHARD III : I 'll be , and my
Train Epoch: 79, Loss: 4.8812, LR: 0.0


## Part 4.G

In [ ]:
inp = tok.encode("").unsqueeze(0).cuda()
print(tok.decode(model.generate(inp, 50)[0].cpu()))